# Imitation Learning — bám làn + đèn tín hiệu (bước 2/3)

Warm-start cho policy network của DRL (bước 3, xem `drl_training/`).

**Input**: mask segmentation 4 lớp (one-hot) + vector số `speed_mps`, `yaw_rate_rps`,
`speed_limit_kmh`, `previous_steer`, `previous_longitudinal`, `traffic_light_state` (one-hot
4 nhãn: green/yellow/red/unknown).
**Output**: `[steer, longitudinal]` ∈ [-1, 1] (longitudinal âm = phanh, dương = ga).

Hai hợp đồng phải giữ đúng, cả hai đều là loại lỗi **im lặng** (không có exception nào):

1. **Không gian nhãn** phải khớp `train-seg.ipynb`, `data_collection/carla_collector/schema.py`
   (`RAW_TO_TRAIN_LANE`) và `drl_training/policy/backbone.py` (`NUM_CLASSES`). Nếu số kênh
   trùng nhau mà bảng nhãn khác nhau thì `load_state_dict` không báo gì — model chỉ đọc sai
   kênh và lái sai. §2b tự đối chiếu với file `.pth` của seg.
2. **Observation** không được chứa `lane_offset_m`, `heading_error_rad`, `is_junction`: đó là
   các đại lượng dùng làm *reward* ở bước DRL. Để lọt vào input thì (a) model học cách đọc
   thẳng sai số thay vì nhìn ảnh (leakage), (b) DRL không có các cột này nên checkpoint
   warm-start lệch shape. Dataset vẫn trả chúng ra riêng dưới tên `aux` để chẩn đoán ở §11.

**Vì sao dữ liệu thu ở 5 FPS.** `previous_steer` là đặc trưng nguy hiểm nhất trong behavior
cloning: ở 20 FPS vô-lăng gần như không kịp đổi trong 50 ms nên `previous_steer ≈ steer`, và
model đạt loss rất thấp bằng cách chép lại nó, bỏ qua hoàn toàn ảnh segmentation. Lúc chạy
thật `previous_steer` là hành động của chính nó ở bước trước → sai số tích luỹ, xe trôi khỏi
làn. Lỗi này **không** hiện ra trong val loss, nên §4 đo baseline "chép lại" và §11 bắt buộc
model phải thắng nó.

Chia train/val theo **town** (Town01–04 / Town05) giống hệt notebook segmentation: Town05
chưa từng xuất hiện lúc train nên MAE sẽ cao hơn cách chia theo session — đó là con số thật.

> **Đã đồng bộ với `train-segment-lane.ipynb`** (lần chạy đạt `lane_mIoU = 0.8917`,
> epoch 17, Unet/ResNet34 + SCSE bias-free, 4 lớp `Background/Road/RoadLine/Sidewalk`
> ở 384×480). Những chỗ đã sửa: `DATASET_ROOT` (tự dò 1–4 tầng mount, mặc định trỏ
> đúng đường dẫn seg đã dùng), số frame kỳ vọng theo TỪNG town (35 000 / 5 000 chứ
> không phải 40 000 / 10 000), cách dò `best_carla_lane_seg.pth`, §3b dựng lại model
> seg theo `decoder_attention_type` + `scse_bias_free` đọc từ chính checkpoint, và
> `NUM_WORKERS`/`persistent_workers` theo đúng bài học rò RAM của notebook seg.
>
> Quyết định duy nhất cần chốt trước khi chạy: `USE_PREDICTED_SEGMENTATION` ở §2.


> **Bản v4.** Tiếp nối v3 (steer MAE 0.0085 → **0.00693**, MAE giữa làn 0.0060 → **0.0029**).
> v3 đã chỉ ra chính xác chỗ hỏng: **13.8% mẫu lệch làn gây 64% tổng sai số** (0.0319 so với
> 0.0029, gấp 10.8 lần). v4 nhắm vào đúng đó:
>
> 1. `SAMPLER_MODE` thêm **`offset`** / `steer+offset` — tăng tần suất frame đã lệch làn,
>    **không cần thu lại dữ liệu**. Thử cái này trước khi đi collect.
> 2. `MERGE_GREEN_INTO_UNKNOWN` — gộp `green` (0.28% mẫu, val có 0) vào `unknown`;
>    `SCALAR_FEATURE_DIM` 9 → 8. Kèm chẩn đoán tốc độ trung bình theo trạng thái đèn để
>    xác định collector có đang bỏ sót đèn xanh không.
> 3. Bỏ `EXPECT_PER_TOWN` hardcode — §4 tự đếm và so với **trung vị** giữa các town.
> 4. §4 báo tỉ lệ dữ liệu recovery; **§12 mới**: kiểm tra sẵn sàng warm-start (ngưỡng
>    baseline, ô one-hot chết, lệch GT↔dự đoán, lỗi đổi dấu) trước khi bàn giao cho DRL.
>
> Val (Town05) **không bao giờ** bị lọc hay cân bằng lại.


## 1. Import

In [ ]:
!pip install segmentation_models_pytorch
import os, glob, random, time
from concurrent.futures import ThreadPoolExecutor

import cv2
# Kaggle GPU session chỉ có 4 vCPU. Để OpenCV tự spawn thread thì nó tranh CPU với
# DataLoader worker và với ThreadPoolExecutor dựng cache mask ở §6.
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. Cấu hình

Mỗi Town có `il_fields.csv` riêng, trong đó `seg_label_path` là đường dẫn **tương đối theo
Town** (vd. `seg_label/00173383.png`). Không có file gộp nào ở gốc dataset — §3 tự gộp cả 5
Town và nối lại thành đường dẫn tuyệt đối.

In [ ]:
def find_town_dirs(root, town):
    """TẤT CẢ folder khớp '{town}*' — một town thu nhiều phiên thì mọi phiên đều được dùng."""
    hits = sorted(h for h in glob.glob(os.path.join(root, f"{town}*")) if os.path.isdir(h))
    if not hits:
        raise FileNotFoundError(f"Không thấy folder '{town}*' trong {root}")
    return hits


def _first_dir_with(patterns, probe):
    """Ứng viên ĐẦU TIÊN vừa tồn tại vừa chứa `probe` bên trong."""
    for pat in patterns:
        for hit in sorted(glob.glob(pat)):
            if os.path.isdir(hit) and glob.glob(os.path.join(hit, probe)):
                return hit
    return None


# --- DATASET_ROOT ------------------------------------------------------------------------
# Lần chạy train-seg.ipynb sinh ra checkpoint đang dùng (lane_mIoU = 0.8917) đọc dữ liệu từ
# SEG_RUN_DATASET_ROOT dưới đây. Kaggle mount cùng một dataset ở độ sâu khác nhau tuỳ cách
# attach, nên dò thêm 1-4 tầng thay vì hardcode một đường dẫn: hai notebook đọc hai bản dữ
# liệu khác nhau là lỗi IM LẶNG, không có exception nào.
SEG_RUN_DATASET_ROOT = "/kaggle/input/datasets/dlocg19/seg-il-dataset/CARLA_DATA"
_ROOT_PATTERNS = [SEG_RUN_DATASET_ROOT,
                  "/kaggle/input/seg-il-dataset/CARLA_DATA",
                  *[f"/kaggle/input/{'*/' * k}CARLA_DATA" for k in range(1, 5)]]
DATASET_ROOT = _first_dir_with(_ROOT_PATTERNS, "Town01*")
if DATASET_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục chứa folder 'Town01*'. Sửa SEG_RUN_DATASET_ROOT.\n"
        f"  Đang mount: {sorted(glob.glob('/kaggle/input/*'))}")
if os.path.realpath(DATASET_ROOT) != os.path.realpath(SEG_RUN_DATASET_ROOT):
    print(f"[!] DATASET_ROOT = {DATASET_ROOT}\n"
          f"    seg đã chạy trên {SEG_RUN_DATASET_ROOT}. Nếu đây là hai bản dữ liệu KHÁC\n"
          f"    nhau thì dừng lại — checkpoint seg sẽ không khớp với dữ liệu IL.")
else:
    print(f"DATASET_ROOT = {DATASET_ROOT}  (đúng bản dữ liệu train-seg đã dùng)")

TRAIN_TOWNS  = ["Town01", "Town02", "Town03", "Town04"]
VAL_TOWNS    = ["Town05"]
TOWN_DIRS    = [(t, d) for t in TRAIN_TOWNS + VAL_TOWNS
                for d in find_town_dirs(DATASET_ROOT, t)]
print("Town dirs:")
for _t, _d in TOWN_DIRS:
    print("  -", _d)

# --- Không gian nhãn: khớp TUYỆT ĐỐI train-seg.ipynb (§2b tự kiểm với file .pth) ---------
CLASS_NAMES = ["Background", "Road", "RoadLine", "Sidewalk"]
NUM_CLASSES = len(CLASS_NAMES)
ROAD_ID     = CLASS_NAMES.index("Road")
ROADLINE_ID = CLASS_NAMES.index("RoadLine")
SKY_ID      = CLASS_NAMES.index("Sky") if "Sky" in CLASS_NAMES else None

# Chỉ 4 raw id được nhắc tên; MỌI raw còn lại (bầu trời 0/13, Vehicles 10, Pedestrian 4,
# Ground 14, Terrain 22, Building, Pole, id lạ > 22...) rơi về Background.
#   Bầu trời -> Background chứ KHÔNG phải ignore: 26% khung hình vẫn được giám sát đầy đủ,
#   chỉ là nó trả lời cùng câu hỏi "không đi được" như tường/cây.
#   Ground -> Background: CARLA định nghĩa Ground là bục/vòng xuyến/sân, không phải mặt
#   đường; gán nó là Road tức là dạy model rằng lề bê tông đi được.
# Bảng này đã được đối chiếu byte-for-byte với `label_lut` trong best_carla_lane_seg.pth
# (seg chạy TASK_SCHEME="lane4", RAILTRACK_AS_ROAD=True, IGNORE_UNLABELED=False,
# SKY_AS_CLASS=False, AUTO_PRUNE không loại class nào) — §2b kiểm lại mỗi lần chạy.
RAW_TO_TRAIN = {
     6: 2,   # RoadLine
     7: 1,   # Road
     8: 3,   # Sidewalk
    16: 1,   # RailTrack -> Road (RAILTRACK_AS_ROAD=True bên seg)
}
SEG_LABEL_LUT = np.zeros(256, dtype=np.uint8)
for _raw, _train in RAW_TO_TRAIN.items():
    SEG_LABEL_LUT[_raw] = _train
del _raw, _train

PALETTE = np.array([
    [ 60,  60,  60],  # 0 Background
    [128,  64, 128],  # 1 Road
    [157, 234,  50],  # 2 RoadLine
    [244,  35, 232],  # 3 Sidewalk
], dtype=np.uint8)

# --- Độ phân giải ------------------------------------------------------------------------
SEG_NATIVE_HEIGHT, SEG_NATIVE_WIDTH = 384, 480   # = IMAGE_HEIGHT/WIDTH của train-seg.ipynb
# IL hạ xuống 1/2 cho rẻ (DRL chạy realtime). AdaptiveAvgPool2d trong SteeringNet khiến
# kích thước này KHÔNG ảnh hưởng shape checkpoint — đặt 384/480 nếu muốn bỏ hẳn resize.
IMAGE_HEIGHT, IMAGE_WIDTH = 192, 240

THIN_COVER_THRESH = 0.25          # = THIN_COVER_THRESH của seg (§2b kiểm lại)

def downscale_labels(lab, out_w=IMAGE_WIDTH, out_h=IMAGE_HEIGHT,
                     thin_ids=(ROADLINE_ID,), thr=THIN_COVER_THRESH):
    """NEAREST cho toàn cục, rồi khôi phục class mảnh bằng độ phủ diện tích.

    NEAREST thuần khi 384->192 xoá phần lớn vạch kẻ (rộng 2-3 px, ~1.6% pixel ở train /
    3.4% ở Town05) — đúng tín hiệu quan trọng nhất cho bám làn. Hàm chạy trên TRAIN ID nên
    dùng được cho CẢ ground-truth lẫn mask dự đoán; bắt buộc phải là cùng một hàm, nếu
    không phân phối lúc train và lúc inference sẽ lệch.
    """
    if lab.shape[0] == out_h and lab.shape[1] == out_w:
        return lab
    out = cv2.resize(lab, (out_w, out_h), interpolation=cv2.INTER_NEAREST)
    for t in thin_ids:
        m = (lab == t)
        if not m.any():
            continue
        cov = cv2.resize(m.astype(np.float32), (out_w, out_h), interpolation=cv2.INTER_AREA)
        out[cov > thr] = t
    return out

# --- Hợp đồng observation với DRL (`drl_training/policy/observation.py`) -----------------
# ==========================================================================================
# RÒ RỈ QUAN SÁT — thay đổi quan trọng nhất của v5. Đọc trước khi động vào danh sách này.
# ==========================================================================================
# `yaw_rate_rps` ĐÃ BỊ LOẠI. Nó không phải một đặc trưng, nó là ĐÁP ÁN: xe đang quay CHÍNH
# VÌ vô-lăng đang quay, và hai đại lượng được đo ở cùng một bước thời gian. Đo trên CARLA
# thật với chính checkpoint v4/v8 (giữ nguyên một khung hình, quét từng đầu vào một):
#
#     nguồn biến thiên       biên độ steer gây ra      so với ảnh
#     yaw_rate_rps           0.368 (v8) / 0.520 (v4)      114x / 290x
#     previous_steer         0.175                         54x
#     ẢNH segmentation       0.0032 (v8) / 0.0018 (v4)      1x
#
# Nhánh CNN gần như không đóng góp gì cho việc đánh lái. Trong vòng kín hậu quả là tất định:
# yaw_rate khởi tạo bằng 0 -> steer ~ 0 -> xe đi thẳng -> yaw_rate vẫn 0. Một vòng lặp tự
# duy trì khiến xe lao thẳng cho tới khi ra khỏi đường — đúng những gì đo được (v4 đi thẳng
# 47m rồi va chạm; v8 còn không rời được vạch xuất phát).
#
# `speed_mps` và `speed_limit_kmh` KHÔNG bị loại: tốc độ là hệ quả của ga TRONG QUÁ KHỨ, và
# là thông tin bắt buộc để quyết định ga hiện tại. Ranh giới là "cùng bước thời gian".
CONTINUOUS_COLS = ["speed_mps", "speed_limit_kmh"]

# Danh sách chốt chặn — §12 kiểm lại. `drl_training/policy/observation.py` giữ đúng bản sao
# và in cảnh báo nếu nạp phải checkpoint cũ.
LEAKY_COLS = ["yaw_rate_rps", "previous_steer", "previous_longitudinal"]

# aux giờ có HAI vai trò khác nhau, đừng lẫn:
#   AUX_COLS       : cột đọc từ CSV, dùng cho chẩn đoán ở §11 (giữ nguyên như v4).
#   AUX_TARGET_COLS: tập con thực sự trở thành MỤC TIÊU HỌC PHỤ ở §8/§9.
AUX_COLS        = ["lane_offset_m", "heading_error_rad", "is_junction"]
# Collector chỉ ghi "green" ở 0.28% mẫu và val (Town05) có ĐÚNG 0 mẫu — trong khi tỉ lệ
# đỏ:xanh hợp lý phải quanh 10:1 (dừng đèn đỏ 10-20s vs qua đèn xanh 1-2s ở 5 FPS). Một ô
# one-hot gần như không được huấn luyện là bẫy im lặng: lúc chạy thật DRL sẽ bật nó lên và
# scalar_mlp trả ra giá trị không có cơ sở. Gộp vào "unknown" cho tới khi collector được sửa
# — cả hai đều mang cùng một nghĩa hành động: "được phép đi".
# Đặt False SAU KHI đợt thu mới cho tỉ số đỏ:xanh hợp lý.
MERGE_GREEN_INTO_UNKNOWN = True
TRAFFIC_LIGHT_VOCAB = (["yellow", "red", "unknown"] if MERGE_GREEN_INTO_UNKNOWN
                       else ["green", "yellow", "red", "unknown"])

# previous_steer / previous_longitudinal là con dao hai lưỡi:
#   Bật -> lệnh lái mượt, nhưng model có thể đạt loss thấp bằng cách phát lại chính đầu
#          vào đó. §11 đo đúng chuyện này bằng baseline "chép".
#   Tắt -> buộc model đọc ảnh segmentation. Mất độ mượt, nhưng PPO ở bước 3 học lại độ mượt
#          qua reward, nên với vai trò WARM-START đây thường mới là lựa chọn đúng.
# CẢNH BÁO HỢP ĐỒNG: tắt cờ này làm SCALAR_FEATURE_DIM 9 -> 7, tức shape của
# `scalar_mlp.0.weight` đổi theo. `drl_training/policy/observation.py` PHẢI dựng vector
# theo đúng `scalar_feature_order` lưu trong checkpoint, đừng chép tay danh sách cột.
# v5: TẮT. Ngoài chuyện là đường tắt cho steer (biên độ 0.175 so với 0.003 của ảnh),
# `previous_longitudinal` còn tạo một điểm hút chết người ở chiều dọc: đo trên CARLA, v8
# xuất phanh ngay bước đầu, giá trị đó quay lại làm đầu vào, và sau 5 bước nó khoá cứng ở
# -0.995. Xe không bao giờ rời vạch xuất phát (0 m trong 2 episode x 150 bước).
USE_PREV_ACTIONS = False
PREV_ACTION_COLS = ["previous_steer", "previous_longitudinal"]    # đã trong [-1,1]
RAW_ACTION_COLS  = PREV_ACTION_COLS if USE_PREV_ACTIONS else []

def normalize_traffic_light(value):
    v = str(value).strip().lower()
    return v if v in TRAFFIC_LIGHT_VOCAB else "unknown"

SCALAR_FEATURE_DIM = len(CONTINUOUS_COLS) + len(RAW_ACTION_COLS) + len(TRAFFIC_LIGHT_VOCAB)
# Thứ tự ghép vector — phía DRL phải build ĐÚNG thứ tự này.
SCALAR_FEATURE_ORDER = CONTINUOUS_COLS + RAW_ACTION_COLS + \
    [f"traffic_light_{v}" for v in TRAFFIC_LIGHT_VOCAB]

# --- Bộ dữ liệu 5 FPS --------------------------------------------------------------------
COLLECT_FPS = 5.0
CONTROL_DT  = 1.0 / COLLECT_FPS     # 0.2 s giữa hai frame liên tiếp
EPISODE_COL = "session_id"
# CONTROL_DT là hợp đồng với DRL, không chỉ là metadata: `previous_*` nghĩa là "lệnh của 1
# bước TRƯỚC", ở 20 FPS bước đó cách 50 ms còn ở 5 FPS cách 200 ms. Đặt fixed_delta_seconds
# của CARLA lúc train DRL đúng bằng CONTROL_DT.

# Không hardcode số frame kỳ vọng nữa. Mỗi đợt thu cho số khác nhau, và một hằng số lỗi
# thời khiến §4 in cảnh báo giả ở mọi lần chạy — người đọc quen tay bỏ qua đúng lúc nó báo
# thật. §4 tự đếm rồi so từng town với TRUNG VỊ: cái cần phát hiện là town LỆCH so với các
# town khác, không phải lệch so với một con số ghi từ tháng trước.
TOWN_IMBALANCE_TOL = 0.25        # lệch quá 25% so với trung vị -> cảnh báo

# --- Dữ liệu hồi phục làn (recovery) ------------------------------------------------------
# Phân tích v3 chỉ thẳng vào đây: 13.8% mẫu val nằm ngoài 0.15m so với tâm làn nhưng đóng
# góp 64% TỔNG sai số (MAE 0.0319 so với 0.0029 ở giữa làn — gấp 10.8 lần). Autopilot luôn
# chạy giữa làn nên model chưa từng học cách quay về, và trong vòng kín thì hễ bắt đầu trôi
# là nó rơi vào đúng vùng nó yếu nhất -> sai số tự khuếch đại.
RECOVERY_OFFSET_THRESH = 0.15    # |lane_offset_m| lớn hơn mức này = "đang lệch làn"

# --- Nhật ký thí nghiệm ------------------------------------------------------------------
# Mỗi lần chạy đổi RUN_NAME; §11 ghi thêm MỘT dòng vào RESULTS_LOG. Nhờ vậy bảng so sánh
# trong báo cáo được tích luỹ tự động thay vì chép tay từ output đã cuộn mất.
RUN_NAME    = "v5_no_leak_aux"
RESULTS_LOG = "/kaggle/working/il_results.csv"

# --- Optimization ------------------------------------------------------------------------
# batch 32 -> 128. SteeringNet chỉ 0.146M tham số và mask nằm sẵn trong RAM nên ở bs=32 GPU
# gần như rảnh. Batch lớn hơn vừa nhanh hơn vừa giảm phương sai gradient — đường val MAE của
# lần chạy trước dao động 0.008-0.025 giữa các epoch, tức NHIỄU CÒN LỚN HƠN khoảng cách tới
# baseline, không thể kết luận gì từ nó. LR scale theo CĂN BẬC HAI tỉ lệ batch (quy ước của
# train-seg.ipynb): batch gấp 4 thì gradient noise chỉ giảm 2 lần.
BASE_LR, BASE_BATCH = 1e-3, 32
BATCH_SIZE     = 128
LEARNING_RATE  = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5
EPOCHS         = 40
WEIGHT_DECAY   = 1e-4
GRAD_CLIP_NORM = 1.0
WARMUP_STEPS   = 200
MIN_LR_RATIO   = 0.02
EARLY_STOP_PATIENCE = 12

STEER_LOSS_WEIGHT = 2.0
LONGITUDINAL_LOSS_WEIGHT = 1.0

# --- Loss phụ: bắt nhánh CNN phải mã hoá vị trí ngang ------------------------------------
# Bỏ rò rỉ mới chỉ CẤM model đi đường tắt; nó không tự tạo ra tín hiệu học từ ảnh. Đây là
# phần TẠO ra tín hiệu đó: buộc nhánh CNN — và chỉ nhánh CNN, xem `aux_head` ở §8, nó cắm
# thẳng vào đặc trưng ảnh và không thấy scalar — dự đoán xe đang lệch bao nhiêu mét so với
# tâm làn và lệch hướng bao nhiêu radian. Hai đại lượng đó CHỈ đọc được từ ảnh, nên gradient
# của chúng không có đường tắt nào để đi.
#
# Nhãn đã sẵn có: collector vẫn luôn ghi `lane_offset_m` và `heading_error_rad`; v4 chỉ dùng
# chúng để in bảng chẩn đoán ở §11. Không phải thu thêm dữ liệu.
#
# Vẫn TUYỆT ĐỐI không đưa hai cột này vào observation — chúng là input của REWARD bên DRL.
# Làm MỤC TIÊU học phụ thì khác hẳn làm ĐẦU VÀO: model phải suy ra chúng từ ảnh.
AUX_TARGET_COLS   = ["lane_offset_m", "heading_error_rad"]
AUX_LOSS_WEIGHT   = 0.5      # 0.0 = tắt hẳn, quay về hành vi v4
# Đưa hai mục tiêu phụ về cùng thang (mét so với radian) trước khi tính loss.
AUX_TARGET_SCALE  = [1.0, 0.3]

# --- Cân bằng thang đo giữa hai chiều hành động ------------------------------------------
# |steer| điển hình 0.005-0.03 còn |longitudinal| chạy cả dải [-1, 1]. Với cùng một beta,
# gradient của chiều longitudinal áp đảo, và "xuất steer ~ 0" trở thành cực tiểu rẻ nhất —
# chính xác thứ v4 rơi vào. Chia sai số mỗi chiều cho độ lệch chuẩn của chính nó (tính trên
# TRAIN ở §8) để hai chiều đóng góp gradient tương đương.
# Không đổi không gian ĐẦU RA -> `drl_training` không phải sửa gì.
NORMALIZE_ACTION_LOSS = True
# 0.15 -> 0.02. §11 chấm điểm bằng MAE, nhưng SmoothL1 là BẬC HAI khi |err| < beta. Với
# beta=0.15, gần như TOÀN BỘ dữ liệu (80% mẫu "đi thẳng", |Δsteer| < 0.007) nằm trong vùng
# gradient tỉ lệ thuận sai số -> sai số nhỏ gần như không tạo áp lực học, model dừng lại ở
# mức "đủ gần". Hạ beta đưa phần lớn mẫu về vùng tuyến tính (gradient hằng số, khớp MAE) mà
# vẫn giữ tính bền của Huber cho khung cua gấp / hồi phục làn.
HUBER_BETA = 0.02

# EMA: trung bình trượt trọng số qua các step. Rẻ (một lần copy 0.146M tham số mỗi step) và
# là cách trực tiếp nhất dập nhiễu val MAE nói trên. Vòng lặp train đánh giá CẢ HAI (raw và
# EMA) rồi giữ bản tốt hơn — giống hệt train-seg.ipynb.
USE_EMA        = True
EMA_DECAY      = 0.999
EMA_EVAL_START = 3        # trước đó EMA còn quá gần khởi tạo, đánh giá nó chỉ tốn thời gian

# --- Cân bằng lại tập TRAIN (val Town05 KHÔNG bao giờ bị đụng vào) -----------------------
#   "none"     : không cân bằng. Phân phối train = phân phối val -> cơ hội cao nhất vượt
#                baseline ở §11. DÙNG CHO LẦN CHẠY ĐẦU để có con số sạch.
#   "steer"    : cân bằng theo mức |steer|. Tốt cho năng lực vào cua thật, nhưng lệch khỏi
#                phân phối val nên MAE (không trọng số) sẽ xấu đi — đó là đánh đổi, không
#                phải lỗi.
#   "steer+tl" : thêm cân bằng theo đèn tín hiệu. Ở bộ dữ liệu hiện tại việc này KHUẾCH ĐẠI
#                frame dừng đèn đỏ (steer=0, brake=-1, gần như trùng lặp) vì "red" bị coi là
#                hiếm trong khi thực ra nó dư thừa — xem cảnh báo ở §3.
#   "offset"     : cân bằng theo |lane_offset_m|. NHẮM THẲNG vào điểm yếu tìm được ở v3 —
#                  tăng tần suất frame ĐÃ lệch làn mà không cần thu thêm dữ liệu. Thử cái
#                  này TRƯỚC khi kết luận là phải đi collect lại.
#   "steer+offset": kết hợp cả hai.
SAMPLER_MODE = "none"
# Độ mạnh cân bằng: w = (1/tần_suất) ** SAMPLER_POWER, rồi chuẩn hoá về trung bình 1.
#   1.0 = nghịch đảo tần suất đầy đủ (bản cũ). Với phân bố 80/12/5/3.5% thì bin "sharp"
#         được lấy mẫu nhiều gấp 23 lần bin "straight" — mỗi batch bị vài chục mẫu cua gấp
#         lặp lại chi phối, đúng nguồn nhiễu của đường val MAE lần trước.
#   0.5 = cân bằng vừa phải, tỉ số còn ~4.8 lần. Đây là mức thường dùng và là mặc định.
#   0.0 = không cân bằng (tương đương SAMPLER_MODE = "none").
SAMPLER_POWER      = 0.5
# Bin dùng cho SAMPLER_MODE có "offset". Ranh giới trùng với bảng chẩn đoán ở §11 để hai
# nơi nói cùng một ngôn ngữ.
OFFSET_BIN_EDGES  = [-1e-9, 0.15, 0.4, 0.8, np.inf]
OFFSET_BIN_LABELS = ["<0.15m", "0.15-0.4m", "0.4-0.8m", ">0.8m"]
SAMPLER_WEIGHT_CAP = 4.0     # trần tính theo "số lần so với bình thường" sau chuẩn hoá

# Lọc chuỗi đứng yên, CHỈ trên train. Ở 5 FPS một đèn đỏ 20 giây = 100 frame trùng khít
# (speed~0, steer=0, longitudinal=-1). Giữ tối đa STATIONARY_KEEP frame liên tiếp là đủ dạy
# "dừng khi đèn đỏ" mà không để chúng chiếm chỗ dữ liệu lái thật.
#
# v5 BẬT cờ này, và lý do đã đổi hẳn so với v4. Sau khi bỏ `previous_longitudinal` khỏi
# observation (§2), tín hiệu chi phối chiều dọc chỉ còn `speed_mps`. Nếu tập train đầy frame
# dừng đèn đỏ thì model học đúng một luật: "speed ~ 0 -> phanh". Mà xe LUÔN spawn ở tốc độ 0.
# Kết quả là một điểm hút y hệt lỗi của v8 (xe không rời vạch xuất phát 0 m trong 150 bước),
# chỉ khác đường dẫn: qua `speed_mps` thay vì qua `previous_longitudinal`.
#
# Giữ STATIONARY_KEEP = 5 frame (1 giây ở 5 FPS) mỗi lần dừng là vẫn đủ dạy "dừng khi đèn
# đỏ", đồng thời giữ được các frame KHỞI HÀNH từ trạng thái đứng yên — thứ mà model bắt buộc
# phải học nếu muốn tự cất bánh. §11c bên dưới đo trực tiếp xem nó có học được không.
DROP_STATIONARY_RUNS = True
STATIONARY_SPEED     = 0.1      # m/s
STATIONARY_KEEP      = 5

# --- Kiến trúc: lưới pooling --------------------------------------------------------------
# (1, 1) = trung bình toàn ảnh mỗi kênh, tức bản đồ đặc trưng 12x15 bị bóp thành một số mỗi
# kênh, nên bố cục TRÁI-PHẢI của làn đường phải đi vòng qua tương quan giữa các kênh. (4, 6)
# giữ lại bố cục thô: 4 hàng (gần -> xa) x 6 cột (trái -> phải).
# PHẢI KHỚP `POOL_GRID` trong drl_training/policy/backbone.py. Lệch là báo lỗi shape ở
# `cnn_fc.0.weight` lúc warm-start PPO — cố ý để nó chết to thay vì chạy im lặng.
POOL_GRID = (4, 6)

# --- Runtime -----------------------------------------------------------------------------
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = torch.cuda.is_available()
N_GPU   = torch.cuda.device_count()
# SteeringNet chỉ 0.15M tham số nên IL KHÔNG bị chặn bởi GPU (xem §8) — GPU thứ hai của
# Kaggle cố tình để trống ở đây. Chỗ duy nhất dùng tới nó là §3b, nơi phải chạy model
# segmentation (24.55M param) trên 40k ảnh.

# Giải nén sẵn toàn bộ mask thành train id ở đúng IMAGE_HEIGHT x IMAGE_WIDTH và giữ trong
# RAM: 1 byte/pixel, ~46 KB/frame (40k frame ~ 1.85 GB trên 33.7 GB của Kaggle). Đổi lại
# không phải decode PNG + remap + resize lại ở MỌI epoch. Kết quả không đổi vì các bước
# này đều tất định.
CACHE_MASKS_IN_RAM = True
# NUM_WORKERS = 0 KHI ĐÃ CACHE. Hai lý do, lý do thứ hai là lý do train-seg.ipynb phải vá:
#   1. __getitem__ chỉ còn index vào mảng numpy trong RAM (~46 KB) — rẻ hơn chi phí pickle
#      + gửi tensor qua hàng đợi IPC tới tiến trình chính.
#   2. `_loader_kwargs` dùng chung cho CẢ train_loader lẫn val_loader. Với workers > 0 +
#      persistent_workers=True thì 4 worker của train sống song song với 4 worker của val
#      trên 4 vCPU, và worker không bao giờ chết cũng là worker không bao giờ trả RSS lại
#      cho OS — bên seg đo được +2.9 GB/epoch tuyến tính, chạm trần RAM ở epoch 5.
NUM_WORKERS = 0 if CACHE_MASKS_IN_RAM else min(os.cpu_count() or 2, 4)
SEG_INFER_WORKERS = min(os.cpu_count() or 2, 4)   # §3b decode JPEG/PNG -> vẫn cần worker

STEER_CHECKPOINT_PATH = "/kaggle/working/best_il_model.pth"

# --- Checkpoint segmentation (đầu vào của §2b và §3b) ------------------------------------
# §2b IM LẶNG bỏ qua kiểm tra hợp đồng nhãn nếu không thấy file — mà trên Kaggle checkpoint
# seg nằm ở /kaggle/working (vừa train xong) hoặc trong dataset gắn thêm ở độ sâu 1-4 tầng,
# không bao giờ ở cwd. Dò cả bốn tầng để chốt chặn quan trọng nhất của notebook không bị
# tắt mà không ai biết.
_SEG_PATTERNS = ["best_carla_lane_seg.pth",
                 "/kaggle/working/best_carla_lane_seg.pth",
                 *[f"/kaggle/input/{'*/' * k}best_carla_lane_seg.pth" for k in range(1, 5)]]
SEG_CHECKPOINT_PATH = next((h for pat in _SEG_PATTERNS for h in sorted(glob.glob(pat))
                            if os.path.isfile(h)), "best_carla_lane_seg.pth")
# True = thiếu checkpoint seg thì DỪNG thay vì train tiếp trong im lặng. Chỉ đặt False khi
# cố tình muốn train thử bằng ground-truth mà chưa có file .pth trong tay.
REQUIRE_SEG_CONTRACT = True

# --- QUYẾT ĐỊNH DUY NHẤT CẦN CHỐT TRƯỚC KHI BẤM RUN --------------------------------------
# False: train trên ground-truth segmentation (nhanh, mask hoàn hảo).
# True : sinh mask bằng chính model seg rồi train trên đó — KHỚP đúng phân phối mà DRL sẽ
#        gặp lúc chạy thật (Sidewalk IoU chỉ 0.81, RoadLine 0.89, tức mask thật KHÁC
#        ground-truth ở đúng những pixel dùng để bám làn). Tốn thêm ~20-30 phút cho 40k ảnh
#        x 2 forward trên 2 GPU và ~300 MB trong /kaggle/working.
# Khuyến nghị: True nếu còn thời gian trong session — đây là nguồn sai lệch train/inference
# lớn nhất của cả chuỗi 3 bước.
USE_PREDICTED_SEGMENTATION = False
PREDICTED_MASK_DIR = "/kaggle/working/predicted_seg_masks"
SEG_PATHS_ARE_TRAIN_IDS = False     # mask dự đoán đã là train id -> không áp LUT lần hai


def seed_worker(worker_id):
    # Thiếu hàm này thì mọi worker dùng chung một chuỗi RNG của random/numpy — nghiêm trọng
    # nhất trên Windows/macOS (spawn), nơi mỗi worker chạy lại random.seed(SEED) ở đầu file.
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)

DATALOADER_GENERATOR = torch.Generator()
DATALOADER_GENERATOR.manual_seed(SEED)

print(f"\n{NUM_CLASSES} class {CLASS_NAMES} | {IMAGE_HEIGHT}x{IMAGE_WIDTH} "
      f"(gốc {SEG_NATIVE_HEIGHT}x{SEG_NATIVE_WIDTH}) | scalar_dim={SCALAR_FEATURE_DIM}")
print(f"batch={BATCH_SIZE} | epochs={EPOCHS} | lr={LEARNING_RATE:.2e} | workers={NUM_WORKERS} "
      f"| cache_mask={CACHE_MASKS_IN_RAM} | amp={USE_AMP} | {N_GPU} GPU")
print(f"RUN_NAME={RUN_NAME} | huber_beta={HUBER_BETA} | ema={USE_EMA} "
      f"| sampler={SAMPLER_MODE}(p={SAMPLER_POWER}, cap {SAMPLER_WEIGHT_CAP}) "
      f"| prev_actions={USE_PREV_ACTIONS} "
      f"| drop_stationary={DROP_STATIONARY_RUNS}")
print(f"kỳ vọng train={EXPECT_TRAIN} ({TRAIN_TOWNS}) | val={EXPECT_VAL} ({VAL_TOWNS})")
print(f"seg ckpt = {SEG_CHECKPOINT_PATH}"
      + ("" if os.path.exists(SEG_CHECKPOINT_PATH) else "   [CHƯA THẤY FILE]"))
print(f"segmentation dùng để train = "
      f"{'DỰ ĐOÁN (§3b)' if USE_PREDICTED_SEGMENTATION else 'GROUND-TRUTH'}")


## 2b. Kiểm tra hợp đồng nhãn với model segmentation

Chốt chặn quan trọng nhất của notebook: bảng nhãn từng bị chép tay ở ba nơi (notebook seg,
notebook IL, `schema.py`) và đã lệch nhau trong thực tế. Cell này đối chiếu với chính file
`.pth` thay vì tin vào comment.

In [ ]:
SEG_META = {}
if os.path.exists(SEG_CHECKPOINT_PATH):
    _sc = torch.load(SEG_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    _names = list(_sc["class_names"])
    _lut = np.array(_sc["label_lut"], dtype=np.int64)

    print(f"seg checkpoint : {SEG_CHECKPOINT_PATH}")
    print(f"  epoch        : {_sc.get('epoch', -1) + 1} "
          f"(trọng số '{_sc.get('weights_source', '?')}')")
    print(f"  lớp          : {_names}")
    print(f"  kiến trúc    : {_sc.get('model_arch')}/{_sc.get('encoder_name')} "
          f"| attention={_sc.get('decoder_attention_type')} "
          f"| scse_bias_free={_sc.get('scse_bias_free')}")
    print(f"  kích thước   : {_sc.get('image_height')}x{_sc.get('image_width')}")
    print(f"  lane_mIoU    : {_sc.get('best_lane_miou', float('nan')):.4f} "
          f"| drivable IoU {_sc.get('iou_drivable', float('nan')):.4f}")
    if _sc.get("per_class_iou"):
        print("  IoU/class    : " + ", ".join(
            f"{n}={v:.3f}" for n, v in zip(_names, _sc["per_class_iou"])))

    assert _names == CLASS_NAMES, (
        f"LỆCH TÊN/THỨ TỰ LỚP.\n  seg: {_names}\n  IL : {CLASS_NAMES}\n"
        "Sửa CLASS_NAMES ở §2 cho khớp seg, ĐỪNG sửa ngược lại.")

    _diff = [(r, int(_lut[r]), int(SEG_LABEL_LUT[r])) for r in range(23)
             if int(_lut[r]) != int(SEG_LABEL_LUT[r])]
    assert not _diff, ("LỆCH LUT tại (raw, seg, IL): " + str(_diff) +
                       "\nThường gặp: seg đang để IGNORE_UNLABELED=True nên raw 0 -> 255, "
                       "hoặc AUTO_PRUNE đã gộp một class về Background.")

    # Bước thời gian: seg không dùng tới nó, nhưng nếu hai notebook đọc hai bản dataset khác
    # nhau thì đây là chỗ lộ ra sớm nhất.
    _fps = _sc.get("collect_fps")
    if _fps is not None and abs(_fps - COLLECT_FPS) > 1e-6:
        raise RuntimeError(f"LỆCH FPS: seg={_fps}, IL={COLLECT_FPS}. "
                           "Hai notebook đang đọc hai bản dataset khác nhau.")
    if bool(_sc.get("sky_as_class")) != ("Sky" in CLASS_NAMES):
        raise RuntimeError(f"Lệch cách xử lý bầu trời: seg sky_as_class="
                           f"{_sc.get('sky_as_class')}, IL {'có' if SKY_ID is not None else 'không có'}"
                           " class 'Sky'. Cả hai phải cùng gộp trời vào Background.")

    # Độ phân giải GỐC phải khớp: seg đọc mask ở 384x480 rồi train ở đó, IL đọc CÙNG file
    # PNG đó rồi hạ xuống IMAGE_HEIGHT/WIDTH. Lệch nghĩa là hai notebook nhìn hai bản mask
    # khác nhau, và §3b sẽ sinh mask ở sai kích thước.
    _sh, _sw = _sc.get("image_height"), _sc.get("image_width")
    if (_sh, _sw) != (SEG_NATIVE_HEIGHT, SEG_NATIVE_WIDTH):
        raise RuntimeError(f"LỆCH ĐỘ PHÂN GIẢI GỐC: seg train ở {_sh}x{_sw}, IL khai báo "
                           f"SEG_NATIVE = {SEG_NATIVE_HEIGHT}x{SEG_NATIVE_WIDTH}. "
                           "Sửa SEG_NATIVE_* ở §2 cho khớp seg.")

    # Cùng ngưỡng bảo tồn class mảnh -> vạch kẻ sống sót qua resize theo cùng một luật.
    _thr = _sc.get("thin_cover_thresh")
    if _thr is not None and abs(_thr - THIN_COVER_THRESH) > 1e-9:
        raise RuntimeError(f"LỆCH thin_cover_thresh: seg={_thr}, IL={THIN_COVER_THRESH}.")

    # §3b cần đúng những key này để dựng lại model; thiếu -> dừng ngay thay vì để
    # load_state_dict báo missing/unexpected key giữa chừng.
    if USE_PREDICTED_SEGMENTATION:
        _need = ["model_state_dict", "model_arch", "encoder_name",
                 "decoder_attention_type", "num_classes"]
        _lack = [k for k in _need if k not in _sc]
        if _lack:
            raise KeyError(f"Checkpoint seg thiếu key {_lack} — không dựng lại được model "
                           "cho §3b. Train lại seg bằng bản notebook mới nhất.")

    SEG_META = {k: _sc[k] for k in (
        "num_classes", "class_names", "model_arch", "encoder_name",
        "decoder_attention_type", "scse_bias_free", "image_height", "image_width",
        "norm_mean", "norm_std") if k in _sc}

    print(f"\nHợp đồng nhãn khớp: LUT, tên lớp, thứ tự index, FPS, độ phân giải gốc và "
          f"ngưỡng class mảnh đều giống nhau.")
    del _sc, _names, _lut, _diff
else:
    _msg = ("Không thấy checkpoint segmentation. Đã dò:\n"
            + "\n".join("    " + p for p in _SEG_PATTERNS))
    if REQUIRE_SEG_CONTRACT or USE_PREDICTED_SEGMENTATION:
        raise FileNotFoundError(
            _msg + "\n  Add output của train-seg.ipynb (best_carla_lane_seg.pth) làm input "
                   "dataset của notebook này.\n  Hoặc đặt REQUIRE_SEG_CONTRACT = False ở §2 "
                   "để train bằng ground-truth mà BỎ QUA kiểm tra hợp đồng nhãn.")
    print("[!] " + _msg)
    print("    Train bằng ground-truth vẫn chạy được, nhưng không ai xác nhận rằng hai model")
    print(f"    dùng chung bảng nhãn {CLASS_NAMES} — đó là loại lỗi im lặng.")


## 3. Nạp dữ liệu

In [ ]:
REQUIRED_COLS = ["session_id", "seg_label_path", "speed_mps", "yaw_rate_rps",
                 "previous_steer", "previous_longitudinal", "lane_offset_m",
                 "heading_error_rad", "speed_limit_kmh", "traffic_light_state",
                 "is_junction", "steer", "longitudinal"]
USED_COLS = set(REQUIRED_COLS) | {"frame"}       # il_fields.csv có ~120 cột, chỉ đọc phần cần


def load_town_il_csv(town_dir, town):
    """Đọc il_fields.csv của một Town và đổi cột đường dẫn ảnh sang đường dẫn tuyệt đối.

    Trong CSV chúng tương đối theo thư mục Town nên chỉ đọc đúng nếu cwd trùng thư mục đó.
    """
    df = pd.read_csv(os.path.join(town_dir, "il_fields.csv"),
                     usecols=lambda c: c in USED_COLS)
    df["seg_label_path"] = town_dir + os.sep + df["seg_label_path"]
    # CSV không có cột rgb_path (chỉ §3b cần) — suy từ "frame", cùng quy ước 8 chữ số.
    df["rgb_path"] = [os.path.join(town_dir, "rgb", f"{int(f):08d}.png") for f in df["frame"]]
    df["town"] = town
    return df


df = pd.concat([load_town_il_csv(d, t) for t, d in TOWN_DIRS], ignore_index=True)
df = df.sort_values([EPISODE_COL, "frame"], kind="mergesort").reset_index(drop=True)

missing = [c for c in REQUIRED_COLS if c not in df.columns]
assert not missing, f"CSV thiếu cột: {missing}. Cột hiện có: {list(df.columns)}"

# --- Bước thời gian THẬT, đo từ dữ liệu chứ không tin COLLECT_FPS ------------------------
# `frame` là bộ đếm frame của CARLA. Ở chế độ async nó nhảy không đều, nhưng TRUNG VỊ của
# khoảng cách vẫn cho biết mỗi mẫu cách nhau bao nhiêu bước sim — đủ để phát hiện việc thu
# ở tần số dày hơn khai báo, thứ biến `previous_steer` thành đường tắt (xem §4).
_gap = df.groupby(EPISODE_COL)["frame"].diff().dropna()
print(f"Khoảng cách frame giữa hai mẫu liên tiếp: trung vị {_gap.median():.0f}, "
      f"p10={_gap.quantile(.1):.0f}, p90={_gap.quantile(.9):.0f}")

# --- Đèn tín hiệu: xem giá trị THÔ trước khi chuẩn hoá -----------------------------------
# normalize_traffic_light() đẩy MỌI giá trị lạ về "unknown" và không báo gì. Nếu collector
# ghi "Green"/"3"/"Off"/số nguyên thì đèn xanh biến mất lặng lẽ vào nhóm unknown, và bảng
# MAE theo đèn ở §11 sẽ trả NaN mà không ai biết vì sao. Đây là chỗ duy nhất thấy sự thật.
_raw_tl = df["traffic_light_state"].astype(str).str.strip().str.lower().value_counts()
print("\ntraffic_light_state THÔ (đã strip + lower):")
print(_raw_tl.to_string())
_outside = [v for v in _raw_tl.index if v not in TRAFFIC_LIGHT_VOCAB]
if _outside:
    print(f"[!] {len(_outside)} giá trị NGOÀI vocab sẽ bị gộp vào 'unknown': {_outside[:10]}")
    print("    Nếu trong đó có dạng khác của 'green' thì sửa normalize_traffic_light ở §2,")
    print("    ĐỪNG để nó rơi vào unknown.")

# Chẩn đoán collector: nếu "unknown" có tốc độ trung bình CAO và tỉ lệ is_junction đáng kể
# thì đèn xanh đang nằm lẫn trong đó, tức collector chỉ ghi trạng thái khi xe giảm tốc.
_probe = (df.assign(_tl=df["traffic_light_state"].astype(str).str.strip().str.lower())
            .groupby("_tl")[["speed_mps", "longitudinal", "is_junction"]].mean())
print("\nTrung bình theo trạng thái đèn THÔ (dùng để đoán lỗi collector):")
print(_probe.round(3).to_string())
_n_red = int((df["traffic_light_state"].astype(str).str.lower() == "red").sum())
_n_grn = int((df["traffic_light_state"].astype(str).str.lower() == "green").sum())
if _n_grn:
    print(f"tỉ số đỏ:xanh = {_n_red/_n_grn:.1f} : 1   (hợp lý ~10:1 ở 5 FPS — dừng đèn đỏ "
          f"10-20s vs qua đèn xanh 1-2s)")

df["traffic_light_state"] = df["traffic_light_state"].map(normalize_traffic_light)
if MERGE_GREEN_INTO_UNKNOWN:
    # normalize_traffic_light đã đẩy "green" về "unknown" vì green không còn trong vocab.
    # Khẳng định lại ở đây để lỗi lộ ra ngay thay vì thành một ô one-hot chết.
    assert "green" not in TRAFFIC_LIGHT_VOCAB
    print(f"\nĐÃ GỘP {_n_grn} mẫu 'green' vào 'unknown' -> vocab {TRAFFIC_LIGHT_VOCAB}, "
          f"scalar_dim={SCALAR_FEATURE_DIM}")
    print("  DRL phải map green -> unknown theo đúng traffic_light_vocab đọc từ checkpoint.")

print(f"\nTổng {len(df)} mẫu, {df[EPISODE_COL].nunique()} session, "
      f"{len(df)/COLLECT_FPS/60:.0f} phút lái ở {COLLECT_FPS:.0f} FPS")
print(df.groupby("town").size().to_string())
print(df["traffic_light_state"].value_counts().to_string())

_green = float((df["traffic_light_state"] == "green").mean()) if not MERGE_GREEN_INTO_UNKNOWN else 1.0
if _green < 0.05:
    print(f"\n[!] Đèn xanh chỉ chiếm {100*_green:.2f}% số mẫu — không thể là phân bố lái xe")
    print("    bình thường (pha xanh dài hơn pha vàng nhiều lần). Gần như chắc là lỗi ở")
    print("    collector khi đọc traffic_light_state. Hệ quả cụ thể:")
    print("      1. one-hot traffic_light_green là input model gần như CHƯA TỪNG THẤY, mà")
    print("         DRL lúc chạy thật sẽ dựng nó thường xuyên -> scalar_mlp trả ra rác.")
    print("      2. Nếu val (Town05) có 0 mẫu xanh thì phần 'đèn tín hiệu' KHÔNG đánh giá")
    print("         được, và mọi kết luận về nó trong báo cáo đều không có cơ sở.")
df.head()


### 3b. (Tuỳ chọn) Dùng segmentation dự đoán thay ground-truth

Chỉ chạy khi `USE_PREDICTED_SEGMENTATION = True`. Suy luận ở đúng độ phân giải seg được
train rồi mới hạ xuống kích thước IL bằng `downscale_labels` — cùng đường đi với
ground-truth ở §6.

In [ ]:
def _strip_scse_bias(module):
    """Bỏ bias của 3 conv 1x1 trong mọi SCSEModule — bản sao `align_scse_` của train-seg.

    train-seg.ipynb §8 phải vá như vậy để tránh "CUDA error: misaligned address" của
    DataParallel (bias numel 1 và 2 làm lệch địa chỉ mọi tensor phía sau trong buffer
    broadcast). Checkpoint vì thế KHÔNG chứa các key bias đó. Dựng smp.Unet mặc định rồi
    load_state_dict sẽ nổ "Missing key(s): decoder.blocks.0.attention1.cSE.1.bias, ...".
    Phải vá y hệt TRƯỚC khi nạp trọng số.
    """
    def no_bias(conv):
        new = nn.Conv2d(conv.in_channels, conv.out_channels, conv.kernel_size,
                        stride=conv.stride, padding=conv.padding,
                        dilation=conv.dilation, groups=conv.groups, bias=False)
        with torch.no_grad():
            new.weight.copy_(conv.weight)
        return new

    n = 0
    for m in module.modules():
        if type(m).__name__ == "SCSEModule":
            m.cSE[1] = no_bias(m.cSE[1])
            m.cSE[3] = no_bias(m.cSE[3])
            m.sSE[0] = no_bias(m.sSE[0])
            n += 1
    return n


if USE_PREDICTED_SEGMENTATION:
    os.makedirs(PREDICTED_MASK_DIR, exist_ok=True)
    _ck = torch.load(SEG_CHECKPOINT_PATH, map_location="cpu", weights_only=False)

    # Đọc kiến trúc THẲNG từ checkpoint thay vì chép tay: dựng sai lớp thì load_state_dict
    # báo thiếu/thừa key. `decoder_attention_type` phải lấy từ file — hardcode "scse" ở đây
    # sẽ hỏng ngay khi seg được train lại với DECODER_ATTENTION = None.
    _arch = _ck.get("model_arch", "unet")
    _enc  = _ck.get("encoder_name", "resnet34")
    _att  = _ck.get("decoder_attention_type", "scse")
    if _arch == "unet":
        seg_model = smp.Unet(encoder_name=_enc, encoder_weights=None,
                             decoder_attention_type=_att, in_channels=3,
                             classes=_ck["num_classes"])
    else:
        seg_model = smp.DeepLabV3Plus(encoder_name=_enc, encoder_weights=None,
                                      encoder_output_stride=8, in_channels=3,
                                      classes=_ck["num_classes"])

    if _ck.get("scse_bias_free"):
        print(f"vá SCSE bias-free: {_strip_scse_bias(seg_model)} module (khớp checkpoint seg)")

    seg_model.load_state_dict(_ck["model_state_dict"])   # strict=True: sai kiến trúc là nổ ngay
    seg_model.to(DEVICE).eval().requires_grad_(False)

    # Đây là bước DUY NHẤT trong notebook đáng dùng cả 2 GPU: 40k ảnh x 2 forward (flip-TTA)
    # bằng ~8 epoch val của notebook segmentation. Suy luận thuần nên DataParallel gần như
    # scale tuyến tính (không có gradient để gom về GPU0).
    seg_net = nn.DataParallel(seg_model) if N_GPU > 1 else seg_model
    SEG_INFER_BATCH = 32 * max(N_GPU, 1)

    _H = _ck.get("image_height", SEG_NATIVE_HEIGHT)
    _W = _ck.get("image_width",  SEG_NATIVE_WIDTH)
    infer_tf = A.Compose([
        A.Resize(height=_H, width=_W, interpolation=cv2.INTER_LINEAR),
        A.Normalize(mean=tuple(_ck.get("norm_mean", (0.485, 0.456, 0.406))),
                    std=tuple(_ck.get("norm_std",  (0.229, 0.224, 0.225)))),
        ToTensorV2(),
    ])
    print(f"seg: {_arch}/{_enc} attention={_att} @ {_H}x{_W}, {_ck['num_classes']} lớp "
          f"{_ck['class_names']} | batch {SEG_INFER_BATCH} trên {max(N_GPU, 1)} GPU")

    # Khoá cache là "session_id + tên file", KHÔNG phải riêng basename: "frame" đếm lại
    # từ đầu mỗi phiên nên các Town có dải trùng nhau.
    _rgb_paths = df["rgb_path"].to_numpy()
    predicted_paths = [
        os.path.join(PREDICTED_MASK_DIR,
                     f"{s}_{os.path.basename(p)}".replace(".png", "_pred.png"))
        for s, p in zip(df[EPISODE_COL], _rgb_paths)]
    todo = [i for i, p in enumerate(predicted_paths) if not os.path.exists(p)]

    class _RgbInferDataset(Dataset):
        """Đọc + chuẩn hoá ảnh trong DataLoader worker, không chặn GPU như vòng lặp bs=1."""

        def __init__(self, indices):
            self.indices = indices

        def __len__(self):
            return len(self.indices)

        def __getitem__(self, k):
            i = self.indices[k]
            img = cv2.imread(_rgb_paths[i], cv2.IMREAD_COLOR)
            if img is None:
                raise FileNotFoundError(_rgb_paths[i])
            return infer_tf(image=cv2.cvtColor(img, cv2.COLOR_BGR2RGB))["image"], i

    if todo:
        # Workers ở đây là CẦN: mỗi mẫu phải decode 1 JPEG/PNG 384x480 + normalize, khác hẳn
        # DataLoader train (§6) vốn chỉ index vào cache trong RAM.
        infer_loader = DataLoader(_RgbInferDataset(todo), batch_size=SEG_INFER_BATCH,
                                  shuffle=False, num_workers=SEG_INFER_WORKERS,
                                  pin_memory=torch.cuda.is_available())
        with torch.no_grad():
            for _imgs, _idx in tqdm(infer_loader, desc="Sinh segmentation dự đoán"):
                _imgs = _imgs.to(DEVICE, non_blocking=True)
                with torch.amp.autocast("cuda", enabled=USE_AMP):
                    _logits = seg_net(_imgs) + torch.flip(
                        seg_net(torch.flip(_imgs, dims=[3])), dims=[3])
                _pred = _logits.argmax(1).to(torch.uint8).cpu().numpy()
                for _j, _i in enumerate(_idx.tolist()):
                    cv2.imwrite(predicted_paths[_i], _pred[_j])
    else:
        print("Tất cả mask dự đoán đã có sẵn — bỏ qua bước suy luận.")

    df["seg_label_path"] = predicted_paths
    SEG_PATHS_ARE_TRAIN_IDS = True
    del seg_model, seg_net, _ck
    torch.cuda.empty_cache()
    print(f"Đã chuyển sang segmentation dự đoán (train id 0..{NUM_CLASSES - 1}) tại "
          f"{PREDICTED_MASK_DIR}.")
else:
    print("Dùng ground-truth segmentation (raw id CARLA -> LUT).")
    print("Lưu ý: DRL lúc chạy thật chỉ có mask DỰ ĐOÁN, nơi Sidewalk IoU = 0.81 và "
          "RoadLine = 0.89.\n  Đặt USE_PREDICTED_SEGMENTATION = True ở §2 để khớp đúng "
          "phân phối đó (+~20-30 phút).")


## 4. Chia train/val theo town + kiểm tra "sao chép hành động"

MAE của baseline `steer = previous_steer` là **ngưỡng phải vượt** ở §11. Nếu model không
thấp hơn hẳn baseline này thì nó chưa học được gì từ ảnh, dù val loss trông đẹp thế nào.

In [ ]:
train_df = df[df["town"].isin(TRAIN_TOWNS)].reset_index(drop=True)
val_df   = df[df["town"].isin(VAL_TOWNS)].reset_index(drop=True)
assert len(train_df) and len(val_df), \
    f"Chia theo town thất bại — town có mặt: {sorted(df['town'].unique())}"

print(f"Chia theo TOWN: train={TRAIN_TOWNS} val={VAL_TOWNS}")
_counts = df.groupby("town").size()
_median = float(_counts.median())
for _t, _n in _counts.items():
    _dev = (_n - _median) / max(_median, 1)
    print(f"  {_t}: {_n} mẫu ({_dev:+.0%} so với trung vị {_median:.0f})"
          + ("   <-- lệch, cân nhắc thu bù" if abs(_dev) > TOWN_IMBALANCE_TOL else ""))

# --- Lọc chuỗi đứng yên: CHỈ trên train --------------------------------------------------
# val KHÔNG bao giờ bị lọc/cân bằng lại. Đó là điều kiện để mọi con số §11 so sánh được
# giữa các lần chạy: chỉ có model thay đổi, thước đo thì đứng yên.
if DROP_STATIONARY_RUNS:
    _still = train_df["speed_mps"].abs() < STATIONARY_SPEED
    # Đánh số từng chuỗi đứng yên liên tiếp trong mỗi session, rồi giữ STATIONARY_KEEP
    # frame đầu của chuỗi — đủ để model học "thấy đèn đỏ thì dừng và GIỮ dừng".
    _grp = (_still != _still.shift()).cumsum()
    _rank = train_df.groupby([EPISODE_COL, _grp]).cumcount()
    _keep = (~_still) | (_rank < STATIONARY_KEEP)
    print(f"\nLọc chuỗi đứng yên (train): bỏ {int((~_keep).sum())} / {len(train_df)} mẫu "
          f"({100*float((~_keep).mean()):.1f}%), giữ tối đa {STATIONARY_KEEP} frame/chuỗi")
    train_df = train_df[_keep].reset_index(drop=True)

print(f"\nTrain: {len(train_df)} mẫu ({train_df[EPISODE_COL].nunique()} session) | "
      f"Val: {len(val_df)} mẫu ({val_df[EPISODE_COL].nunique()} session)")
print("\nPhân bố đèn tín hiệu (train / val):")
print(pd.DataFrame({
    "train": train_df["traffic_light_state"].value_counts(normalize=True),
    "val":   val_df["traffic_light_state"].value_counts(normalize=True),
}).reindex(TRAFFIC_LIGHT_VOCAB).fillna(0).round(4).to_string())
if not (val_df["traffic_light_state"] == "green").any():
    print("[!] Val KHÔNG có mẫu đèn xanh nào -> dòng 'green' ở bảng MAE §11 sẽ là NaN.")

# --- Dữ liệu hồi phục: có bao nhiêu frame ĐÃ lệch làn để học cách quay về? --------------
for _name, _part in (("train", train_df), ("val", val_df)):
    _off = _part["lane_offset_m"].abs()
    _share = float((_off > RECOVERY_OFFSET_THRESH).mean())
    print(f"[{_name}] {100*_share:5.1f}% mẫu lệch >{RECOVERY_OFFSET_THRESH}m "
          f"({int((_off > RECOVERY_OFFSET_THRESH).sum())} frame) | "
          f"p95 |offset| = {_off.quantile(.95):.3f}m | max = {_off.max():.3f}m")
if float((train_df["lane_offset_m"].abs() > RECOVERY_OFFSET_THRESH).mean()) < 0.05:
    print("[!] Dưới 5% dữ liệu train nằm ngoài tâm làn. Model sẽ giỏi khi đang đi đúng và")
    print("    lạc lối ngay khi trôi — trong vòng kín đó là sai số TỰ KHUẾCH ĐẠI.")
    print("    Hai lối ra: SAMPLER_MODE = 'offset' (dùng lại dữ liệu sẵn có), hoặc thu thêm")
    print("    pha recovery (đặt xe lệch 0.3-0.8m rồi ghi lại lúc autopilot lái về).")

# --- BA baseline. Cả ba đều đo trên val, không học gì cả ---------------------------------
# 1. "chép"      : steer_t = previous_steer. Trả lời "model có học được gì từ ẢNH không, hay
#                  chỉ phát lại đầu vào?". Đây là ngưỡng QUYẾT ĐỊNH checkpoint có dùng được.
# 2. "hằng số"   : steer = 0 / longitudinal = trung bình train. Trả lời "model có học được
#                  gì KHÔNG?". Là mốc dễ — vượt nó không phải thành tựu, nhưng KHÔNG vượt
#                  nổi thì model vô dụng hoàn toàn.
# Hằng số cho longitudinal lấy từ TRAIN (không phải val) — cùng nguyên tắc chống rò rỉ như
# norm_stats ở §5; dùng trung bình val là tự cho baseline xem trước đáp án.
COPYCAT_STEER_MAE = float((val_df["steer"] - val_df["previous_steer"]).abs().mean())
COPYCAT_LONG_MAE  = float((val_df["longitudinal"] - val_df["previous_longitudinal"]).abs().mean())
CONST_STEER_MAE   = float(val_df["steer"].abs().mean())
_long_const       = float(train_df["longitudinal"].mean())
CONST_LONG_MAE    = float((val_df["longitudinal"] - _long_const).abs().mean())

# --- Tuong quan RO RI: do ngay tren du lieu, khong can model -----------------------------
# `yaw_rate_rps` bi loai khoi observation o §2. Bang nay cho thay VI SAO: neu corr(steer,
# yaw_rate) cao thi bat ky model nao duoc phep nhin yaw_rate deu se hoc doc no thay vi nhin
# anh — va MAE offline se rat dep trong khi vong kin that bai hoan toan.
print("\nTuong quan giua steer va cac dai luong DO CUNG BUOC THOI GIAN:")
for _col in LEAKY_COLS:
    if _col not in val_df.columns:
        continue
    _rt = float(train_df["steer"].corr(train_df[_col]))
    _rv = float(val_df["steer"].corr(val_df[_col]))
    _flag = "  <- RO RI MANH" if abs(_rv) > 0.5 else ""
    print(f"  corr(steer, {_col:<22}) train {_rt:+.4f} | val {_rv:+.4f}{_flag}")
print("  Cac cot tren deu KHONG duoc nam trong observation (§2 LEAKY_COLS).")

for name, part in (("train", train_df), ("val", val_df)):
    r = part["steer"].corr(part["previous_steer"])
    mae = (part["steer"] - part["previous_steer"]).abs().mean()
    print(f"\n[{name}] corr(steer, previous_steer) = {r:.4f}"
          f"   MAE baseline 'chép' = {mae:.4f}")

print(f"\n{'':<16}{'chép prev':>12}{'hằng số':>12}")
print(f"{'Steer':<16}{COPYCAT_STEER_MAE:>12.4f}{CONST_STEER_MAE:>12.4f}")
print(f"{'Longitudinal':<16}{COPYCAT_LONG_MAE:>12.4f}{CONST_LONG_MAE:>12.4f}   "
      f"(hằng số = {_long_const:.3f})")
print("\n§11 phải THẮNG cột 'chép prev'. Cột 'hằng số' chỉ để thấy độ khó cơ bản của bài "
      "toán\nvà để báo cáo không bị đọc thành 'model không học được gì'.")
if val_df["steer"].corr(val_df["previous_steer"]) > 0.98:
    print("[!] corr > 0.98: bước thời gian quá dày, baseline 'chép' gần như bất khả chiến "
          "bại.\n    Lấy thưa dữ liệu hoặc đặt USE_PREV_ACTIONS = False ở §2.")


## 5. Chuẩn hoá đặc trưng số

Z-score fit **chỉ trên train** để tránh rò rỉ dữ liệu.

In [ ]:
norm_stats = {c: (float(train_df[c].mean()), float(train_df[c].std()) + 1e-6)
              for c in CONTINUOUS_COLS}
NORM_MEAN = np.array([norm_stats[c][0] for c in CONTINUOUS_COLS], dtype=np.float32)
NORM_STD  = np.array([norm_stats[c][1] for c in CONTINUOUS_COLS], dtype=np.float32)
for c, (m, s) in norm_stats.items():
    print(f"{c}: mean={m:.4f} std={s:.4f}")

## 6. Dataset & DataLoader

Lật ngang kèm đảo dấu đồng bộ `steer`, `previous_steer`, `yaw_rate_rps` (và `lane_offset_m`,
`heading_error_rad` của `aux`).

Dataset trả **class-id map** (H, W) chứ không phải one-hot: 1 byte/pixel thay vì 4 float, và
one-hot được làm trên GPU ngay trước conv — giống hệt `drl_training/policy/backbone.py`.

In [ ]:
def load_mask(path):
    """PNG -> train id ở kích thước IL. THỨ TỰ BẮT BUỘC: remap trước, hạ mẫu sau.

    Resize trước rồi mới remap thì vạch kẻ đã bị NEAREST xoá từ lúc còn là raw id.
    """
    # IMREAD_UNCHANGED chứ không phải IMREAD_GRAYSCALE — giống `read_mask_raw` của
    # train-seg.ipynb. Collector ghi seg_label 1 kênh, nhưng nếu một phiên nào đó ghi ra
    # ảnh 3 kênh thì tag nằm ở kênh đỏ; GRAYSCALE sẽ trộn ba kênh theo trọng số độ sáng và
    # sinh ra raw id KHÔNG TỒN TẠI — mà mọi id lạ đều lặng lẽ rơi về Background qua LUT.
    mask = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if mask is None:
        raise FileNotFoundError(path)
    if mask.ndim == 3:
        mask = np.ascontiguousarray(mask[:, :, 2])
    if not SEG_PATHS_ARE_TRAIN_IDS:
        mask = SEG_LABEL_LUT[mask]
    mask = downscale_labels(mask)
    if mask.max() >= NUM_CLASSES:
        raise ValueError(f"train id {mask.max()} >= NUM_CLASSES={NUM_CLASSES} tại {path} "
                         "— LUT hoặc checkpoint seg không khớp.")
    return mask


class SteeringDataset(Dataset):
    def __init__(self, frame, augment, cache=CACHE_MASKS_IN_RAM):
        frame = frame.reset_index(drop=True)
        self.augment = augment
        self.use_prev = len(RAW_ACTION_COLS) > 0
        # Rút sẵn thành mảng numpy: df.iloc[idx] tốn ~50 us/mẫu, nhân với vài triệu lượt đọc.
        self.paths  = frame["seg_label_path"].to_numpy()
        self.cont   = frame[CONTINUOUS_COLS].to_numpy(np.float32)   # RAW, chuẩn hoá sau khi lật
        self.prev   = frame[RAW_ACTION_COLS].to_numpy(np.float32)   # (N, 0) nếu tắt prev
        self.target = frame[["steer", "longitudinal"]].to_numpy(np.float32)
        self.aux    = frame[AUX_COLS].to_numpy(np.float32)
        self.tl = np.stack([(frame["traffic_light_state"] == v).to_numpy(np.float32)
                            for v in TRAFFIC_LIGHT_VOCAB], axis=1)
        # v5 da bo yaw_rate_rps khoi CONTINUOUS_COLS (xem §2), nen cot nay co the khong
        # ton tai. Giu nhanh xu ly de van chay duoc neu ai do bat lai no de doi chung.
        self.yaw_col = (CONTINUOUS_COLS.index("yaw_rate_rps")
                        if "yaw_rate_rps" in CONTINUOUS_COLS else None)

        self.masks = None
        if cache:
            # Đọc + remap + hạ mẫu 40k PNG mất ~6 phút nếu chạy tuần tự, và nó nằm TRƯỚC
            # mọi epoch nên là phần thời gian chết lớn nhất của notebook. cv2.imread/resize
            # đều nhả GIL -> ThreadPoolExecutor ăn đủ 4 vCPU của Kaggle mà không phải pickle
            # gì (khác multiprocessing). pool.map giữ THỨ TỰ nên i vẫn khớp self.paths.
            self.masks = np.empty((len(frame), IMAGE_HEIGHT, IMAGE_WIDTH), np.uint8)
            n_thread = max(os.cpu_count() or 2, 1)
            with ThreadPoolExecutor(max_workers=n_thread) as pool:
                for i, m in enumerate(tqdm(pool.map(load_mask, self.paths),
                                           total=len(self.paths),
                                           desc="cache mask", leave=False)):
                    self.masks[i] = m
            print(f"  cache mask: {self.masks.nbytes/1e9:.2f} GB cho {len(frame)} frame "
                  f"({n_thread} thread)")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        mask = self.masks[i] if self.masks is not None else load_mask(self.paths[i])
        cont, prev, aux = self.cont[i].copy(), self.prev[i].copy(), self.aux[i].copy()
        steer, longitudinal = self.target[i]

        if self.augment and random.random() < 0.5:
            mask = mask[:, ::-1]
            steer = -steer
            if self.yaw_col is not None:
                cont[self.yaw_col] = -cont[self.yaw_col]
            aux[0], aux[1] = -aux[0], -aux[1]
            if self.use_prev:                 # prev rỗng khi USE_PREV_ACTIONS = False
                prev[0] = -prev[0]

        scalar = np.concatenate([(cont - NORM_MEAN) / NORM_STD, prev, self.tl[i]])
        return (torch.from_numpy(np.ascontiguousarray(mask)),
                torch.from_numpy(scalar.astype(np.float32)),
                torch.from_numpy(aux),
                torch.tensor([steer, longitudinal]))


train_ds = SteeringDataset(train_df, augment=True)
val_ds   = SteeringDataset(val_df, augment=False)
assert train_ds[0][1].shape[0] == SCALAR_FEATURE_DIM, (
    f"scalar vector dài {train_ds[0][1].shape[0]} nhưng SCALAR_FEATURE_DIM="
    f"{SCALAR_FEATURE_DIM} — checkpoint sẽ ghi sai hợp đồng cho DRL.")

if CACHE_MASKS_IN_RAM and os.name == "nt" and NUM_WORKERS > 0:
    NUM_WORKERS = 0
    print("[!] Windows + cache mask -> NUM_WORKERS = 0 (tránh nhân bản cache theo worker)")

# persistent_workers CHỈ bật cho train_loader. Dùng chung một dict cho cả hai loader là đúng
# cái làm train-seg rò +2.9 GB/epoch: worker của train không bao giờ chết trong khi worker
# của val được tạo thêm mỗi epoch. Với NUM_WORKERS = 0 (mặc định khi cache mask) đây là no-op.
_common = dict(num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
               generator=DATALOADER_GENERATOR,
               worker_init_fn=seed_worker if NUM_WORKERS > 0 else None)
if NUM_WORKERS > 0:
    _common["prefetch_factor"] = 4
_train_kwargs = dict(_common, persistent_workers=NUM_WORKERS > 0)
_val_kwargs   = dict(_common, persistent_workers=False)


def _class_weights(labels, power=None, cap=None):
    """w = (1/tần_suất) ** power, chuẩn hoá về trung bình 1, rồi cắt trần.

    Chuẩn hoá để `cap` có nghĩa tuyệt đối: sau bước này 1.0 = "lấy mẫu như bình thường",
    cap = "nhiều nhất bấy nhiêu lần bình thường". `power` mới là knob chính — nghịch đảo
    tần suất đầy đủ (power=1) đẩy tỉ số straight/sharp lên 23 lần, quá mạnh cho một tập chỉ
    có 3.5% mẫu cua gấp.
    """
    power = SAMPLER_POWER if power is None else power
    cap = SAMPLER_WEIGHT_CAP if cap is None else cap
    freq = labels.value_counts(normalize=True)
    w = (1.0 / labels.astype(object).map(freq).to_numpy(np.float64)) ** power
    w /= w.mean()
    return np.clip(w, None, cap)


steer_bins = pd.cut(train_df["steer"].abs(), bins=[-0.001, 0.02, 0.1, 0.3, np.inf],
                    labels=["straight", "gentle", "moderate", "sharp"])
print("Phân bố bin |steer| (train):")
print(steer_bins.value_counts(normalize=True).round(3).to_string())

offset_bins = pd.cut(train_df["lane_offset_m"].abs(), bins=OFFSET_BIN_EDGES,
                     labels=OFFSET_BIN_LABELS)
print("Phân bố bin |lane_offset_m| (train):")
print(offset_bins.value_counts(normalize=True).reindex(OFFSET_BIN_LABELS).round(4).to_string())

if SAMPLER_MODE == "none":
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              drop_last=True, **_train_kwargs)
    print("Sampler: KHÔNG cân bằng — phân phối train giống val, cơ hội cao nhất vượt "
          "baseline ở §11.")
else:
    _parts = SAMPLER_MODE.split("+")
    _known = {"steer", "tl", "offset"}
    if not set(_parts) <= _known:
        raise ValueError(f"SAMPLER_MODE lạ: {SAMPLER_MODE} (hợp lệ: {sorted(_known)})")
    w = np.ones(len(train_df), dtype=np.float64)
    if "steer" in _parts:
        w *= _class_weights(steer_bins)
    if "offset" in _parts:
        # Đây là knob nhắm thẳng vào phát hiện của v3: 13.8% mẫu lệch làn gây 64% sai số.
        w *= _class_weights(offset_bins)
    if "tl" in _parts:
        w *= _class_weights(train_df["traffic_light_state"])
        print("[!] 'tl' coi 'red' là hiếm, trong khi frame dừng đèn đỏ lại là nhóm TRÙNG")
        print("    LẶP nhất (steer=0, brake=-1). Cân nhắc DROP_STATIONARY_RUNS=True.")
    w = np.clip(w / w.mean(), None, SAMPLER_WEIGHT_CAP)

    sampler = WeightedRandomSampler(w, num_samples=len(train_df), replacement=True,
                                    generator=DATALOADER_GENERATOR)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              drop_last=True, **_train_kwargs)
    print(f"Sampler '{SAMPLER_MODE}' (power={SAMPLER_POWER}): trọng số "
          f"[{w.min():.2f}, {w.max():.2f}], tỉ số {w.max()/max(w.min(),1e-9):.1f}x "
          f"(trần {SAMPLER_WEIGHT_CAP}). Lưu ý: phân phối train giờ KHÁC val, nên MAE "
          f"không trọng số ở §11 sẽ xấu đi — đó là đánh đổi có chủ đích.")

# shuffle=False + drop_last=False: §11 dựa vào việc thứ tự batch val trùng thứ tự val_df.
val_loader = DataLoader(val_ds, batch_size=max(BATCH_SIZE, 256), shuffle=False, **_val_kwargs)
print(f"train {len(train_ds)} mẫu / {len(train_loader)} batch | "
      f"val {len(val_ds)} mẫu / {len(val_loader)} batch")


## 7. Trực quan hoá dữ liệu mẫu

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i in range(3):
    lab, scalar, aux, target = train_ds[random.randrange(len(train_ds))]
    lab = lab.numpy()
    axes[i].imshow(PALETTE[lab])
    # % RoadLine để kiểm tra mắt thường rằng vạch kẻ SỐNG SÓT qua bước hạ mẫu 384->192.
    axes[i].set_title(f"steer={target[0]:.2f} long={target[1]:.2f} | "
                      f"RoadLine {100*(lab==ROADLINE_ID).mean():.2f}%")
    axes[i].axis("off")
handles = [plt.Rectangle((0, 0), 1, 1, fc=PALETTE[i] / 255) for i in range(NUM_CLASSES)]
fig.legend(handles, CLASS_NAMES, ncol=NUM_CLASSES, loc="lower center", frameon=False)
plt.tight_layout(); plt.show()
print("RoadLine ở ground-truth gốc chiếm ~1.06% pixel. Xuống dưới ~0.6% nghĩa là "
      "downscale_labels chưa chạy đúng — vạch kẻ đang bị bước hạ mẫu ăn mất.")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(train_df["steer"], bins=50); axes[0].set_title("Phân bố Steer (train)")
axes[1].hist(train_df["longitudinal"], bins=50); axes[1].set_title("Phân bố Longitudinal (train)")
train_df["traffic_light_state"].value_counts().reindex(TRAFFIC_LIGHT_VOCAB).plot(
    kind="bar", ax=axes[2], title="Traffic Light State (train)")
plt.tight_layout(); plt.show()

## 8. Model — 2 nhánh (CNN + MLP)

In [ ]:
import copy, math

# Thang đo của hai chiều hành động, tính TRÊN TRAIN (cùng nguyên tắc chống rò rỉ như
# norm_stats ở §5). Dùng để cân bằng gradient giữa steer và longitudinal — xem `control_loss`.
ACTION_STD = np.array([
    float(train_df["steer"].std()) + 1e-6,
    float(train_df["longitudinal"].std()) + 1e-6,
], dtype=np.float32)
AUX_TARGET_IDX = [AUX_COLS.index(c) for c in AUX_TARGET_COLS]
AUX_SCALE_T = torch.tensor(AUX_TARGET_SCALE, dtype=torch.float32, device=DEVICE)
ACTION_STD_T = torch.tensor(ACTION_STD, device=DEVICE)

print(f"std hành động trên train: steer {ACTION_STD[0]:.4f} | long {ACTION_STD[1]:.4f} "
      f"(tỉ số {ACTION_STD[1]/ACTION_STD[0]:.1f}x)")
print(f"mục tiêu phụ: {AUX_TARGET_COLS} (cột aux {AUX_TARGET_IDX}), trọng số {AUX_LOSS_WEIGHT}")


class SteeringNet(nn.Module):
    """CNN (segmentation) + MLP (scalar) -> [steer, longitudinal], kèm ĐẦU RA PHỤ.

    Kiến trúc và TÊN thuộc tính (`conv`, `pool`, `cnn_fc`, `scalar_mlp`, `head`) phải khớp
    `drl_training/policy/backbone.py` + `actor_critic.py` — đó là nơi actor PPO nạp lại
    trọng số warm-start từ checkpoint này.

    Hai thay đổi của v5 so với v4, cả hai đều phải đồng bộ sang `backbone.py`:

    1. `pool` là AdaptiveAvgPool2d(POOL_GRID) chứ không còn (1, 1). Trung bình toàn ảnh bóp
       bản đồ đặc trưng 12x15 thành một số mỗi kênh; lưới 4x6 giữ lại bố cục gần-xa và
       trái-phải, tức thứ mà bài toán bám làn thực sự cần.

    2. `aux_head` — nhánh MỚI, chỉ có ở IL, KHÔNG mang sang DRL. Nó cắm thẳng vào `x_img`
       (đặc trưng ảnh, TRƯỚC khi ghép với scalar) và phải dự đoán `lane_offset_m` +
       `heading_error_rad`. Vì nó không thấy scalar, cách duy nhất để giảm loss phụ là mã
       hoá vị trí ngang vào chính đặc trưng ảnh — đúng thứ v4 chưa bao giờ học.
       `policy/il_compat.py` lọc theo tiền tố nên `aux_head.*` bị bỏ qua khi warm-start,
       không cần làm gì thêm.
    """

    def __init__(self, num_classes=NUM_CLASSES, num_scalar_features=SCALAR_FEATURE_DIM,
                 pool_grid=POOL_GRID, num_aux=len(AUX_TARGET_COLS)):
        super().__init__()
        self.num_classes = num_classes
        self.conv = nn.Sequential(
            nn.Conv2d(num_classes, 24, 5, 2, 2), nn.BatchNorm2d(24), nn.ELU(),
            nn.Conv2d(24, 36, 5, 2, 2), nn.BatchNorm2d(36), nn.ELU(),
            nn.Conv2d(36, 48, 5, 2, 2), nn.BatchNorm2d(48), nn.ELU(),
            nn.Conv2d(48, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.ELU(),
            nn.Conv2d(64, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ELU(),
        )
        self.pool = nn.AdaptiveAvgPool2d(pool_grid)
        self.cnn_fc = nn.Sequential(nn.Linear(64 * pool_grid[0] * pool_grid[1], 64), nn.ELU())
        self.scalar_mlp = nn.Sequential(nn.Linear(num_scalar_features, 32), nn.ELU(),
                                        nn.Linear(32, 32), nn.ELU())
        self.head = nn.Sequential(
            nn.Linear(64 + 32, 64), nn.ELU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(0.2),
            nn.Linear(32, 2),
        )
        # Không Dropout ở nhánh phụ: nó là công cụ định hình đặc trưng, không phải đầu ra
        # cuối cùng, và nhiễu ở đây chỉ làm loãng chính tín hiệu ta đang cố tạo ra.
        self.aux_head = nn.Sequential(nn.Linear(64, 32), nn.ELU(), nn.Linear(32, num_aux))

    def image_features(self, seg_map):
        if seg_map.dim() == 3:
            seg_map = F.one_hot(seg_map.long(), self.num_classes).permute(0, 3, 1, 2).float()
        return self.cnn_fc(self.pool(self.conv(seg_map)).flatten(1))

    def forward(self, seg_map, scalar_features, return_aux=False):
        """seg_map: (N, H, W) class-id map, hoặc (N, num_classes, H, W) one-hot float."""
        x_img = self.image_features(seg_map)
        x_sca = self.scalar_mlp(scalar_features)
        action = torch.tanh(self.head(torch.cat([x_img, x_sca], dim=1)))
        if return_aux:
            return action, self.aux_head(x_img)
        return action


def control_loss(pred, target, pred_aux=None, target_aux=None):
    """SmoothL1 với HUBER_BETA nhỏ ~ MAE có làm mượt quanh 0, cộng loss phụ.

    Vì sao chia cho ACTION_STD (v5): |steer| điển hình 0.005-0.03 còn |longitudinal| chạy cả
    dải [-1, 1]. Dùng chung một `beta` nghĩa là chiều longitudinal áp đảo gradient, và
    "xuất steer ≈ 0" trở thành cực tiểu rẻ nhất — chính xác thứ v4 rơi vào. Chia mỗi chiều
    cho độ lệch chuẩn của nó đưa cả hai về cùng thang, và `beta` cũng thành tương đối.

    Không đổi không gian ĐẦU RA (model vẫn xuất steer thật, đã tanh) nên `drl_training`
    không phải sửa gì.
    """
    if NORMALIZE_ACTION_LOSS:
        err = (pred - target) / ACTION_STD_T
        steer = F.smooth_l1_loss(err[:, 0], torch.zeros_like(err[:, 0]), beta=HUBER_BETA)
        longitudinal = F.smooth_l1_loss(err[:, 1], torch.zeros_like(err[:, 1]), beta=HUBER_BETA)
    else:
        steer = F.smooth_l1_loss(pred[:, 0], target[:, 0], beta=HUBER_BETA)
        longitudinal = F.smooth_l1_loss(pred[:, 1], target[:, 1], beta=HUBER_BETA)
    loss = STEER_LOSS_WEIGHT * steer + LONGITUDINAL_LOSS_WEIGHT * longitudinal

    if pred_aux is not None and AUX_LOSS_WEIGHT > 0:
        loss = loss + AUX_LOSS_WEIGHT * F.smooth_l1_loss(
            pred_aux / AUX_SCALE_T, target_aux / AUX_SCALE_T, beta=0.1)
    return loss


def val_score(steer_mae, long_mae):
    """Điểm CHỌN CHECKPOINT. Cùng trọng số với loss nhưng tính trên MAE.

    Cũng chuẩn hoá theo ACTION_STD như `control_loss`, nếu không thì điểm chọn checkpoint và
    hàm đang tối ưu lại nói hai ngôn ngữ khác nhau.
    """
    if NORMALIZE_ACTION_LOSS:
        steer_mae = steer_mae / float(ACTION_STD[0])
        long_mae = long_mae / float(ACTION_STD[1])
    return STEER_LOSS_WEIGHT * steer_mae + LONGITUDINAL_LOSS_WEIGHT * long_mae


class ModelEMA:
    """Trung bình trượt trọng số. decay tăng dần để những step đầu không bị khởi tạo kéo lại."""

    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval().requires_grad_(False)
        self.decay, self.step = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.step += 1
        d = min(self.decay, (1 + self.step) / (10 + self.step))
        for e, m in zip(self.module.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point:
                e.mul_(d).add_(m.detach(), alpha=1.0 - d)
            else:
                e.copy_(m)          # num_batches_tracked của BatchNorm là int


model = SteeringNet().to(DEVICE)
ema = ModelEMA(model, EMA_DECAY) if USE_EMA else None

# Bias/BatchNorm không chịu weight decay (thực hành chuẩn của AdamW).
decay, no_decay = [], []
for _n, _p in model.named_parameters():
    if _p.requires_grad:
        (no_decay if _p.ndim <= 1 or _n.endswith(".bias") else decay).append(_p)
optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY},
                               {"params": no_decay, "weight_decay": 0.0}], lr=LEARNING_RATE)

_TOTAL_STEPS = max(EPOCHS * len(train_loader), 1)


def _lr_lambda(step):
    if step < WARMUP_STEPS:
        return (step + 1) / WARMUP_STEPS
    p = min((step - WARMUP_STEPS) / max(1, _TOTAL_STEPS - WARMUP_STEPS), 1.0)
    return MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * p))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

_n_img = sum(p.numel() for n, p in model.named_parameters()
             if n.startswith(("conv.", "cnn_fc.")))
print(f"Số tham số: {sum(p.numel() for p in model.parameters())/1e6:.3f}M "
      f"(nhánh ảnh {_n_img/1e6:.3f}M) | pool {POOL_GRID} "
      f"| scalar_dim={SCALAR_FEATURE_DIM} {SCALAR_FEATURE_ORDER}")
print(f"{_TOTAL_STEPS:,} step | warmup {WARMUP_STEPS} | lr {LEARNING_RATE:.2e} -> "
      f"{LEARNING_RATE*MIN_LR_RATIO:.2e} | huber_beta={HUBER_BETA} | ema={USE_EMA}")
print(f"chuẩn hoá loss theo std hành động: {NORMALIZE_ACTION_LOSS}")

# Chốt chặn khớp kiến trúc với DRL: shape của `cnn_fc.0.weight` phụ thuộc POOL_GRID, và nếu
# hai bên lệch thì lỗi chỉ lộ ra lúc warm-start PPO, sau khi đã train xong cả tiếng.
assert model.cnn_fc[0].in_features == 64 * POOL_GRID[0] * POOL_GRID[1]
print(f"cnn_fc nhận {model.cnn_fc[0].in_features} chiều — drl_training/policy/backbone.py "
      f"phải có POOL_GRID = {POOL_GRID}")


## 9. Vòng lặp huấn luyện

In [ ]:
def run_epoch(loader, train, net=None):
    """Trả về (loss trung bình theo MẪU, MAE theo MẪU cho [steer, longitudinal], MAE phụ).

    Cộng theo MẪU chứ không theo batch: val_loader không drop_last nên batch cuối nhỏ hơn,
    tính trọng số ngang các batch đầy sẽ làm MAE lệch nhẹ và không so được với baseline vốn
    tính trên toàn bộ mẫu.

    v5: kéo thêm `aux` ra khỏi loader (v4 vứt nó đi bằng `_aux`) và cho model dự đoán nó.
    MAE phụ được trả về để §10 vẽ được — nếu đường này KHÔNG giảm thì nhánh CNN vẫn chưa học
    được gì về vị trí ngang, và mọi con số steer đẹp đẽ đều đáng ngờ.
    """
    net = model if net is None else net
    net.train(train)
    tot_loss, sum_abs, sum_aux_abs, n = 0.0, np.zeros(2), np.zeros(len(AUX_TARGET_COLS)), 0
    use_aux = AUX_LOSS_WEIGHT > 0
    with torch.enable_grad() if train else torch.no_grad():
        for masks, scalars, aux, targets in loader:
            masks = masks.to(DEVICE, non_blocking=True)
            scalars = scalars.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            aux_t = aux[:, AUX_TARGET_IDX].to(DEVICE, non_blocking=True)
            bs = targets.size(0)

            with torch.amp.autocast("cuda", enabled=USE_AMP):
                if use_aux:
                    preds, preds_aux = net(masks, scalars, return_aux=True)
                    loss = control_loss(preds, targets, preds_aux, aux_t)
                else:
                    preds, preds_aux = net(masks, scalars), None
                    loss = control_loss(preds, targets)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer); scaler.update()
                scheduler.step()            # lịch theo STEP, xem §8
                if ema is not None:
                    ema.update(net)

            tot_loss += loss.item() * bs
            sum_abs += (preds - targets).abs().sum(dim=0).detach().float().cpu().numpy()
            if preds_aux is not None:
                sum_aux_abs += (preds_aux - aux_t).abs().sum(dim=0).detach().float().cpu().numpy()
            n += bs
    return tot_loss / n, sum_abs / n, sum_aux_abs / n


In [ ]:
# `val_aux_mae` là đường quan trọng nhất để theo dõi trong v5: nó đo trực tiếp xem nhánh CNN
# có đang học vị trí ngang không. Nếu nó PHẲNG trong khi `val_steer_mae` vẫn giảm thì model
# đang giảm loss bằng đường khác, không phải bằng cách nhìn ảnh.
HKEYS = ("train_loss", "val_loss", "val_steer_mae", "val_long_mae", "val_score",
         "val_aux_mae", "weights_src", "epoch_time", "lr")
history = {k: [] for k in HKEYS}
best_score, best_src, epochs_no_improve = float("inf"), "raw", 0
training_start = time.time()


def save_checkpoint(epoch, net, src, steer_mae, long_mae, score):
    torch.save({
        "epoch": epoch, "model_state_dict": net.state_dict(), "weights_source": src,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "val_score": score, "val_steer_mae": steer_mae, "val_long_mae": long_mae,
        # --- Hợp đồng nhãn + tiền xử lý + đặc trưng, để DRL không phải chép tay ---------
        "num_classes": NUM_CLASSES, "class_names": CLASS_NAMES,
        "seg_label_lut": SEG_LABEL_LUT.tolist(),
        "road_id": ROAD_ID, "roadline_id": ROADLINE_ID, "sky_id": SKY_ID,
        "image_height": IMAGE_HEIGHT, "image_width": IMAGE_WIDTH,
        "seg_native_height": SEG_NATIVE_HEIGHT, "seg_native_width": SEG_NATIVE_WIDTH,
        "thin_cover_thresh": THIN_COVER_THRESH,
        "pool_grid": list(POOL_GRID),
        "aux_target_cols": AUX_TARGET_COLS, "aux_loss_weight": float(AUX_LOSS_WEIGHT),
        "aux_target_scale": list(AUX_TARGET_SCALE),
        "normalize_action_loss": bool(NORMALIZE_ACTION_LOSS),
        "action_std": [float(ACTION_STD[0]), float(ACTION_STD[1])],
        "leaky_cols_excluded": LEAKY_COLS,
        "scalar_feature_dim": SCALAR_FEATURE_DIM,
        "scalar_feature_order": SCALAR_FEATURE_ORDER,
        "continuous_cols": CONTINUOUS_COLS, "raw_action_cols": RAW_ACTION_COLS,
        "use_prev_actions": bool(USE_PREV_ACTIONS),
        "norm_stats": norm_stats, "traffic_light_vocab": TRAFFIC_LIGHT_VOCAB,
        "collect_fps": COLLECT_FPS, "control_dt": CONTROL_DT,
        "train_towns": TRAIN_TOWNS, "val_towns": VAL_TOWNS,
        # --- Baseline đi kèm để không bao giờ phải nhớ lại "ngưỡng là bao nhiêu" -------
        "copycat_steer_mae": COPYCAT_STEER_MAE, "copycat_long_mae": COPYCAT_LONG_MAE,
        "const_steer_mae": CONST_STEER_MAE, "const_long_mae": CONST_LONG_MAE,
        # --- Siêu tham số của lần chạy, để đối chiếu với il_results.csv ----------------
        "run_name": RUN_NAME, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
        "huber_beta": HUBER_BETA, "sampler_mode": SAMPLER_MODE,
        "sampler_power": SAMPLER_POWER, "sampler_weight_cap": SAMPLER_WEIGHT_CAP,
        "drop_stationary_runs": bool(DROP_STATIONARY_RUNS),
        "seg_checkpoint": os.path.basename(SEG_CHECKPOINT_PATH),
        "trained_on_predicted_seg": bool(USE_PREDICTED_SEGMENTATION),
    }, STEER_CHECKPOINT_PATH)


pbar = tqdm(range(EPOCHS), desc="Training IL")
for epoch in pbar:
    t0 = time.time()
    lr_now = optimizer.param_groups[0]["lr"]
    train_loss, _, _ = run_epoch(train_loader, True)
    val_loss, val_mae, val_aux_mae = run_epoch(val_loader, False)
    if epoch == 0:
        print("  [hợp đồng] scalar_feature_order = %s" % SCALAR_FEATURE_ORDER)

    # Ứng viên: trọng số raw và trọng số EMA. Đánh giá EMA là MỘT lượt val đầy đủ nữa, chỉ
    # đáng làm sau khi trung bình trượt đã rời xa khởi tạo (EMA_EVAL_START).
    cands = [("raw", val_mae, val_loss)]
    if ema is not None and epoch >= EMA_EVAL_START:
        ema_loss, ema_mae, _ = run_epoch(val_loader, False, net=ema.module)
        cands.append(("ema", ema_mae, ema_loss))
    src, mae, vloss = min(cands, key=lambda c: val_score(c[1][0], c[1][1]))
    score = val_score(mae[0], mae[1])

    for k, v in zip(HKEYS, (train_loss, vloss, mae[0], mae[1], score,
                            float(np.mean(val_aux_mae)), src, time.time() - t0, lr_now)):
        history[k].append(v)
    pbar.set_postfix(train=f"{train_loss:.4f}", steer=f"{mae[0]:.4f}",
                     long=f"{mae[1]:.4f}", aux=f"{np.mean(val_aux_mae):.4f}", src=src)

    if score < best_score - 1e-6:
        best_score, best_src, epochs_no_improve = score, src, 0
        save_checkpoint(epoch, model if src == "raw" else ema.module,
                        src, float(mae[0]), float(mae[1]), float(score))
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Dừng sớm ở epoch {epoch+1}")
            break

_ck = torch.load(STEER_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
print(f"\nHoàn tất sau {(time.time()-training_start)/60:.1f} phút "
      f"({np.mean(history['epoch_time']):.1f}s/epoch) -> {STEER_CHECKPOINT_PATH}")
print(f"Best: epoch {_ck['epoch']+1} ({_ck['weights_source']}) "
      f"| steer MAE {_ck['val_steer_mae']:.4f} (baseline chép {COPYCAT_STEER_MAE:.4f}) "
      f"| long MAE {_ck['val_long_mae']:.4f} (baseline chép {COPYCAT_LONG_MAE:.4f})")
print(f"EMA thắng ở {sum(s == 'ema' for s in history['weights_src'])}/"
      f"{len(history['weights_src'])} epoch")
del _ck


> Checkpoint lưu kèm `norm_stats`, `scalar_feature_order`, `traffic_light_vocab` và
> `seg_label_lut` — lúc inference/demo phải nạp lại đúng các giá trị này, không tính lại từ
> dữ liệu mới. `drl_training/policy/observation.py` đọc thẳng các trường này để dựng scalar
> vector đúng thứ tự khi warm-start actor PPO.

## 10. Biểu đồ huấn luyện

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_title("Loss theo epoch"); axes[0].set_xlabel("epoch")
axes[0].legend(); axes[0].grid(alpha=.3)

# Hai đường ngang là thứ duy nhất cần nhìn: đường xanh phải chui xuống DƯỚI đường đỏ.
axes[1].plot(history["val_steer_mae"], label="Steer MAE", color="tab:blue")
axes[1].axhline(COPYCAT_STEER_MAE, color="r", ls=":", lw=1.4, label="chép prev (steer)")
axes[1].axhline(CONST_STEER_MAE, color="gray", ls="--", lw=1.0, label="hằng số (steer)")
axes[1].set_yscale("log")      # baseline hằng số cách baseline chép cả một bậc độ lớn
axes[1].set_title("Steer MAE vs baseline (Val, log)"); axes[1].set_xlabel("epoch")
axes[1].legend(fontsize=8); axes[1].grid(alpha=.3, which="both")

axes[2].plot(history["val_long_mae"], label="Longitudinal MAE", color="tab:orange")
axes[2].axhline(COPYCAT_LONG_MAE, color="r", ls=":", lw=1.4, label="chép prev (long)")
axes[2].axhline(CONST_LONG_MAE, color="gray", ls="--", lw=1.0, label="hằng số (long)")
axes[2].set_yscale("log")
axes[2].set_title("Longitudinal MAE vs baseline (Val, log)"); axes[2].set_xlabel("epoch")
axes[2].legend(fontsize=8); axes[2].grid(alpha=.3, which="both")

plt.tight_layout(); plt.show()

# Đường aux tách riêng: nó không cùng đơn vị với MAE hành động nên vẽ chung sẽ vô nghĩa.
# Đây là đường phải nhìn TRƯỚC khi tin vào val_steer_mae — xem chú thích ở §9.
if AUX_LOSS_WEIGHT > 0 and any(history["val_aux_mae"]):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(history["val_aux_mae"], color="tab:green")
    ax.set_title("MAE mục tiêu phụ (nhánh CNN suy ra vị trí ngang)")
    ax.set_xlabel("epoch"); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    _first, _last = history["val_aux_mae"][0], history["val_aux_mae"][-1]
    print(f"aux MAE: {_first:.4f} -> {_last:.4f} "
          f"({100*(1-_last/max(_first,1e-9)):+.0f}%)")
    if _last > 0.8 * _first:
        print("[!] aux MAE gần như không giảm — nhánh CNN VẪN chưa học được vị trí ngang.")
        print("    Tăng AUX_LOSS_WEIGHT, hoặc kiểm tra lane_offset_m trong CSV có đúng không.")

# Epoch nào EMA thắng trọng số raw — nếu gần như mọi epoch thì nhiễu step-to-step còn lớn.
_ema_ep = [i + 1 for i, s in enumerate(history["weights_src"]) if s == "ema"]
print(f"Thời gian trung bình/epoch: {np.mean(history['epoch_time']):.1f}s "
      f"| LR cuối: {history['lr'][-1]:.2e}")
print(f"EMA được chọn ở epoch: {_ema_ep if _ema_ep else 'không epoch nào'}")


## 11. Đánh giá trên tập Validation

Gồm scatter dự đoán–thực tế, phân bố sai số, MAE theo trạng thái đèn (kiểm tra model có
phản ứng đúng lúc đèn đỏ không — đây là nhãn hiếm), MAE theo mức lệch làn, và đối chiếu
với baseline "chép `previous_steer`".

In [ ]:
ckpt = torch.load(STEER_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
eval_model = SteeringNet(num_scalar_features=ckpt["scalar_feature_dim"]).to(DEVICE)
eval_model.load_state_dict(ckpt["model_state_dict"]); eval_model.eval()
print(f"Đánh giá checkpoint epoch {ckpt['epoch']+1} ({ckpt['weights_source']}) "
      f"| run='{ckpt['run_name']}'")

all_preds, all_targets, all_aux = [], [], []
with torch.no_grad():
    for masks, scalars, aux, targets in tqdm(val_loader, desc="eval val"):
        preds = eval_model(masks.to(DEVICE), scalars.to(DEVICE))
        all_preds.append(preds.float().cpu().numpy())
        all_targets.append(targets.numpy()); all_aux.append(aux.numpy())
all_preds   = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)
all_aux     = np.concatenate(all_aux)      # cột: lane_offset_m, heading_error_rad, is_junction
errors = all_preds - all_targets
# val_loader không shuffle và không drop_last -> thứ tự trùng val_df.
all_tl   = val_df["traffic_light_state"].to_numpy()[:len(all_preds)]
prev_st  = val_df["previous_steer"].to_numpy()[:len(all_preds)]
mae_final = np.abs(errors).mean(axis=0)

# =========================================================================================
# 1. BẢNG BA CỘT — con số chính của báo cáo
# =========================================================================================
# Baseline nào là ngưỡng "đạt" phụ thuộc vào việc model CÓ ĐƯỢC NHÌN previous_* hay không.
#   có  -> "chép prev" là đối thủ công bằng, và là ngưỡng quyết định (hành vi v4).
#   không (v5) -> baseline đó dùng một đầu vào model không hề có. Ngưỡng công bằng là baseline
#                 HẰNG SỐ (dự đoán 0 cho steer), tức câu hỏi "model có học được gì từ ảnh không".
_HAS_PREV = len(ckpt["raw_action_cols"]) > 0
_GATE_MAE = ((COPYCAT_STEER_MAE, COPYCAT_LONG_MAE) if _HAS_PREV
             else (CONST_STEER_MAE, CONST_LONG_MAE))
print("\n" + "=" * 74)
print(f"{'':<14}{'model':>10}{'chép prev':>12}{'hằng số':>10}{'vs chép':>10}{'vs hằng số':>12}")
for i, (lb, cc, cst) in enumerate((("Steer", COPYCAT_STEER_MAE, CONST_STEER_MAE),
                                   ("Longitudinal", COPYCAT_LONG_MAE, CONST_LONG_MAE))):
    g1 = 100 * (1 - mae_final[i] / cc) if cc > 0 else float("nan")
    g2 = 100 * (1 - mae_final[i] / cst) if cst > 0 else float("nan")
    print(f"{lb:<14}{mae_final[i]:>10.4f}{cc:>12.4f}{cst:>10.4f}"
          f"{g1:>+9.1f}%{g2:>+11.1f}%   " + ("ĐẠT" if mae_final[i] < _GATE_MAE[i] else "KHÔNG ĐẠT"))
print("=" * 74)
if _HAS_PREV:
    print("'chép prev' là ngưỡng QUYẾT ĐỊNH. 'hằng số' chỉ cho thấy độ khó cơ bản của bài toán.")
else:
    print("Cột 'ĐẠT/KHÔNG ĐẠT' chấm theo baseline HẰNG SỐ, không theo 'chép prev'.")
    print("Lý do: v5 đã bỏ previous_steer khỏi observation (§2), nên so model với một baseline")
    print("ĐƯỢC nhìn previous_steer là so người bị bịt mắt với người được xem đáp án. Thua nó")
    print("là bình thường và không nói lên điều gì. Ngưỡng thật sự nằm ở §11b: ẢNH phải là")
    print("nguồn chi phối steer.")

# =========================================================================================
# 2. MODEL THẮNG/THUA Ở ĐÂU — chia theo mức thay đổi thật của vô-lăng
# =========================================================================================
# Sai số của baseline "chép" trên MỖI mẫu chính xác bằng |steer - previous_steer|. Chia
# theo đại lượng đó cho thấy điều mà MAE trung bình che mất: baseline gần như hoàn hảo khi
# xe đi thẳng, và sụp đổ khi vô-lăng thật sự đổi — đúng lúc bộ điều khiển cần đúng.
_d = np.abs(all_targets[:, 0] - prev_st)
cmp_tab = pd.DataFrame({
    "bin": pd.cut(_d, [-1e-9, 0.005, 0.02, 0.05, np.inf],
                  labels=["<0.005", "0.005-0.02", "0.02-0.05", ">0.05"]),
    "model": np.abs(errors[:, 0]),
    "chép prev": _d,
}).groupby("bin", observed=False).agg(
    n=("model", "size"), model=("model", "mean"), copy_=("chép prev", "mean"))
cmp_tab["model thắng"] = np.where(cmp_tab["model"] < cmp_tab["copy_"], "có", "không")
print("\nMAE steer theo |steer - previous_steer| (sai số của baseline chính là cột này):")
print(cmp_tab.rename(columns={"copy_": "chép prev"}).round(4).to_string())

# =========================================================================================
# 3. LỖI NGUY HIỂM — thứ MAE trung bình không nhìn thấy
# =========================================================================================
# Một frame "phanh gấp mà model đạp ga" có |err| ~ 1.5 nhưng nếu chỉ chiếm 1% mẫu thì nó
# chỉ đóng góp 0.015 vào MAE — chìm nghỉm. Trong vòng kín thì đó là một va chạm.
gt_l, pr_l = all_targets[:, 1], all_preds[:, 1]
brake_as_throttle = int(((gt_l < -0.1) & (pr_l > 0.1)).sum())
throttle_as_brake = int(((gt_l > 0.1) & (pr_l < -0.1)).sum())
sign_flip_steer = int(((np.abs(all_targets[:, 0]) > 0.05) &
                       (np.sign(all_preds[:, 0]) != np.sign(all_targets[:, 0]))).sum())
N = len(all_preds)
print("\nLỗi nguy hiểm (không hiện ra trong MAE):")
print(f"  GT phanh  -> model ĐẠP GA : {brake_as_throttle:>5} / {N}  "
      f"({100*brake_as_throttle/N:.2f}%)   <-- nguy hiểm nhất")
print(f"  GT ga     -> model PHANH  : {throttle_as_brake:>5} / {N}  "
      f"({100*throttle_as_brake/N:.2f}%)")
print(f"  steer NGƯỢC dấu (|gt|>.05): {sign_flip_steer:>5} / {N}  "
      f"({100*sign_flip_steer/N:.2f}%)")
print(f"  steer p99 = {np.percentile(np.abs(errors[:,0]), 99):.4f} "
      f"| max = {np.abs(errors[:,0]).max():.4f}")
print(f"  long  p99 = {np.percentile(np.abs(errors[:,1]), 99):.4f} "
      f"| max = {np.abs(errors[:,1]).max():.4f}")

# =========================================================================================
# 4. Biểu đồ
# =========================================================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for i, lb in enumerate(["Steer", "Longitudinal"]):
    axes[i].scatter(all_targets[:, i], all_preds[:, i], alpha=0.25, s=8)
    lims = [min(all_targets[:, i].min(), all_preds[:, i].min()),
            max(all_targets[:, i].max(), all_preds[:, i].max())]
    axes[i].plot(lims, lims, "r--", lw=1)
    if i == 1:      # tô hai góc phần tư "đổi dấu" — vùng gây va chạm
        axes[i].axhspan(0.1, lims[1], xmin=0, xmax=0.45, color="red", alpha=0.06)
    axes[i].set_xlabel("Thực tế"); axes[i].set_ylabel("Dự đoán"); axes[i].set_title(lb)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i, lb in enumerate(["Steer", "Longitudinal"]):
    axes[i].hist(errors[:, i], bins=60)
    axes[i].set_yscale("log")     # đuôi hiếm mới là phần đáng lo, thang tuyến tính giấu nó
    axes[i].set_title(f"Phân bố sai số {lb} (log)")
plt.tight_layout(); plt.show()

mae_by_tl = pd.DataFrame({"tl": all_tl, "steer_ae": np.abs(errors[:, 0]),
                          "long_ae": np.abs(errors[:, 1])})
mae_by_tl = mae_by_tl.groupby("tl")[["steer_ae", "long_ae"]].mean().reindex(TRAFFIC_LIGHT_VOCAB)
print("\nMAE theo trạng thái đèn:")
print(mae_by_tl.round(4).to_string())
if mae_by_tl["steer_ae"].isna().any():
    print("[!] Trạng thái đèn có NaN = val không có mẫu nào. Không kết luận gì về nó trong "
          "báo cáo.")

# aux KHÔNG vào model, chỉ dùng ở đây. Kỳ vọng: steer_ae tăng khi |lane_offset_m| lớn — đó
# là vùng phải lái mạnh để hồi phục, và là vùng cần thêm dữ liệu recovery/DAgger.
_off = np.abs(all_aux[:, 0])
mae_by_offset = pd.DataFrame({
    "bin": pd.cut(_off, bins=[-1e-9, 0.15, 0.4, 0.8, np.inf],
                  labels=["<0.15m", "0.15-0.4m", "0.4-0.8m", ">0.8m"]),
    "steer_ae": np.abs(errors[:, 0]),
}).groupby("bin", observed=False)["steer_ae"].agg(["mean", "count"])
print("\nMAE steer theo |lane_offset_m| (chỉ để chẩn đoán):")
print(mae_by_offset.round(4).to_string())
RECOVERY_MAE = float(np.abs(errors[_off > 0.15, 0]).mean()) if (_off > 0.15).any() else float("nan")
CENTER_MAE   = float(np.abs(errors[_off <= 0.15, 0]).mean())

# =========================================================================================
# §11c. XE CÓ TỰ CẤT BÁNH ĐƯỢC KHÔNG — điểm chết của vòng kín
# =========================================================================================
# Mọi episode đều bắt đầu ở tốc độ 0. Nếu model phản xạ "đứng yên -> phanh" thì nó không bao
# giờ rời vạch xuất phát, dù bám làn giỏi đến đâu. Đây chính xác là cách v8 hỏng, và sau khi
# v5 bỏ `previous_longitudinal` thì `speed_mps` là đường dẫn còn lại cho đúng lỗi đó.
_val_speed = val_df["speed_mps"].to_numpy()[:len(all_preds)]
_slow = _val_speed < 0.5
if _slow.any():
    _gt_slow = all_targets[_slow, 1]
    _pr_slow = all_preds[_slow, 1]
    print("\n§11c. Hành vi khi xe gần như đứng yên (speed < 0.5 m/s, %d/%d mẫu val):"
          % (int(_slow.sum()), len(all_preds)))
    print(f"  longitudinal THỰC TẾ : trung bình {{_gt_slow.mean():+.3f}} "
          f"(ga {{100*float((_gt_slow > 0.1).mean()):.0f}}% / phanh {{100*float((_gt_slow < -0.1).mean()):.0f}}%)")
    print(f"  longitudinal DỰ ĐOÁN : trung bình {{_pr_slow.mean():+.3f}} "
          f"(ga {{100*float((_pr_slow > 0.1).mean()):.0f}}% / phanh {{100*float((_pr_slow < -0.1).mean()):.0f}}%)")
    STARTS_FROM_STOP = bool((_pr_slow > 0.05).mean() > 0.3)
    if not STARTS_FROM_STOP:
        print("  [!] Model gần như KHÔNG bao giờ đạp ga khi đang đứng yên. Trong vòng kín xe")
        print("      sẽ không rời được vạch xuất phát. Kiểm tra DROP_STATIONARY_RUNS (§2) —")
        print("      frame dừng đèn đỏ nhiều khả năng vẫn đang lấn át frame khởi hành.")
    else:
        print("  Model có đạp ga khi đứng yên -> xe sẽ tự cất bánh được.")
else:
    STARTS_FROM_STOP = True
    print("\n§11c. Tập val không có mẫu nào đứng yên — bỏ qua kiểm tra cất bánh.")
print(f"MAE giữa làn (<0.15m) = {CENTER_MAE:.4f} | MAE lệch làn (>0.15m) = {RECOVERY_MAE:.4f}"
      f"  -> gấp {RECOVERY_MAE/max(CENTER_MAE,1e-9):.1f} lần")
print(f"Chỉ {100*float((_off > 0.15).mean()):.1f}% mẫu val nằm ngoài 0.15m — autopilot luôn "
      "chạy giữa làn,\nnên model chưa từng học cách quay về. Đây là chỗ cần dữ liệu recovery.")

# =========================================================================================
# 5. Ghi một dòng vào nhật ký thí nghiệm
# =========================================================================================
_row = {
    "run": RUN_NAME, "time": pd.Timestamp.now().strftime("%m-%d %H:%M"),
    "n_train": len(train_df), "n_val": len(val_df),
    "batch": BATCH_SIZE, "lr": round(LEARNING_RATE, 6), "huber_beta": HUBER_BETA,
    "sampler": SAMPLER_MODE, "sampler_power": SAMPLER_POWER,
    "prev_actions": USE_PREV_ACTIONS,
    "drop_stationary": DROP_STATIONARY_RUNS, "pred_seg": USE_PREDICTED_SEGMENTATION,
    "best_epoch": int(ckpt["epoch"]) + 1, "src": ckpt["weights_source"],
    "steer_mae": round(float(mae_final[0]), 5),
    "long_mae": round(float(mae_final[1]), 5),
    "copy_steer": round(COPYCAT_STEER_MAE, 5), "copy_long": round(COPYCAT_LONG_MAE, 5),
    "const_steer": round(CONST_STEER_MAE, 5), "const_long": round(CONST_LONG_MAE, 5),
    "beats_copy": bool(mae_final[0] < COPYCAT_STEER_MAE),
    # `beats_gate` moi la cot de doc: no cham theo baseline PHU HOP voi hop dong observation
    # (xem _GATE_MAE o phan dau cell). `beats_copy` giu lai chi de so voi cac lan chay v4.
    "gate": "copy" if _HAS_PREV else "const",
    "beats_gate": bool(mae_final[0] < _GATE_MAE[0]),
    # Cau hinh kien truc/loss — de bang ablation trong bao cao tu giai thich duoc.
    "pool_grid": "x".join(str(v) for v in POOL_GRID),
    "aux_w": AUX_LOSS_WEIGHT,
    "norm_loss": NORMALIZE_ACTION_LOSS,
    "leak_free": not any(c in LEAKY_COLS for c in CONTINUOUS_COLS + RAW_ACTION_COLS),
    "brake_as_throttle": brake_as_throttle,
    "recovery_mae": round(RECOVERY_MAE, 5), "center_mae": round(CENTER_MAE, 5),
}
_log = pd.DataFrame([_row])
if os.path.exists(RESULTS_LOG):
    _log = pd.concat([pd.read_csv(RESULTS_LOG), _log], ignore_index=True)
_log.to_csv(RESULTS_LOG, index=False)
print(f"\nĐã ghi kết quả vào {RESULTS_LOG} ({len(_log)} lần chạy):")
_cols = [c for c in ["run", "leak_free", "pool_grid", "aux_w", "steer_mae", "gate",
                     "beats_gate", "long_mae", "brake_as_throttle"] if c in _log.columns]
print(_log[_cols].to_string(index=False))
if not _HAS_PREV:
    print("\n('beats_gate' chấm theo baseline hằng số, không theo 'chép prev' — xem §11 ở trên.")
    print(" Các dòng cũ từ v4 có thể thiếu cột mới; pandas điền NaN, không sao.)")

# =========================================================================================
# 6. Kết luận
# =========================================================================================
print("\n" + "=" * 74)
if mae_final[0] < COPYCAT_STEER_MAE and mae_final[1] < COPYCAT_LONG_MAE:
    print(f"ĐẠT — model IL sẵn sàng warm-start cho DRL: {STEER_CHECKPOINT_PATH}")
    print(f"  Nhớ: DRL phải dựng scalar vector đúng thứ tự {SCALAR_FEATURE_ORDER}")
    print(f"  và đặt fixed_delta_seconds = {CONTROL_DT} ({COLLECT_FPS:.0f} FPS).")
else:
    print("[!] CHƯA vượt baseline 'chép'. Thử theo thứ tự, mỗi lần ĐỔI MỘT thứ:")
    print("    1. SAMPLER_MODE = 'none'        (nếu đang bật) — bỏ lệch phân phối train/val")
    print("    2. USE_PREV_ACTIONS = False     — bỏ hẳn đường tắt, buộc model đọc ảnh")
    print("       (đổi SCALAR_FEATURE_DIM 9 -> 7, phải cập nhật đồng bộ bên DRL)")
    print("    3. DROP_STATIONARY_RUNS = True  — bỏ frame trùng lặp lúc dừng đèn đỏ")
    print("    Đổi RUN_NAME mỗi lần; il_results.csv sẽ gom thành bảng so sánh cho báo cáo.")
print("=" * 74)


## 11b. Nguồn thông tin nào điều khiển model (chốt chặn chống rò rỉ)

Xáo trộn từng nguồn đầu vào rồi đo MAE xấu đi bao nhiêu. Nếu xáo **ảnh** mà MAE steer gần như không đổi thì model không dùng ảnh để lái — bất kể MAE ở §11 thấp đến đâu. Đây chính là phép kiểm tra mà v4 thiếu.


In [ ]:
# =========================================================================================
# §11b. ĐẦU VÀO NÀO THỰC SỰ ĐIỀU KHIỂN MODEL — chốt chặn mới của v5
# =========================================================================================
# Đây là phép kiểm tra mà v4 KHÔNG có, và vì thế v4 xuất xưởng với một model không lái được
# trong khi mọi con số ở §11 đều đẹp.
#
# Cách đo (permutation importance): xáo trộn MỘT nguồn thông tin trên toàn tập val rồi xem
# MAE xấu đi bao nhiêu. Nếu xáo ảnh mà MAE steer gần như không đổi, model không dùng ảnh —
# bất kể MAE thấp đến đâu.
#
# Ngưỡng đọc kết quả: nguồn nào làm MAE steer tăng NHIỀU NHẤT chính là nguồn model đang dựa
# vào. Với một policy bám làn đúng nghĩa, nguồn đó BẮT BUỘC phải là ảnh.
rng = np.random.RandomState(SEED)


def _mae_with(perm_scalar_idx=None, perm_image=False):
    """MAE [steer, long] trên val khi xáo một cột scalar, hoặc xáo thứ tự ảnh."""
    tot, n = np.zeros(2), 0
    with torch.no_grad():
        for masks, scalars, _aux, targets in val_loader:
            masks, scalars = masks.to(DEVICE), scalars.to(DEVICE)
            bs = targets.size(0)
            if perm_image:
                # Ghép ảnh của mẫu KHÁC với scalar của mẫu này -> phá liên kết ảnh <-> nhãn.
                masks = masks[torch.randperm(bs, device=DEVICE)]
            if perm_scalar_idx is not None:
                scalars = scalars.clone()
                scalars[:, perm_scalar_idx] = scalars[torch.randperm(bs, device=DEVICE),
                                                      perm_scalar_idx]
            preds = eval_model(masks, scalars).float().cpu().numpy()
            tot += np.abs(preds - targets.numpy()).sum(axis=0)
            n += bs
    return tot / n


base_mae = _mae_with()
rows = [("(không xáo)", base_mae[0], base_mae[1], 0.0)]
img_mae = _mae_with(perm_image=True)
rows.append(("ẢNH segmentation", img_mae[0], img_mae[1], img_mae[0] - base_mae[0]))
for _i, _name in enumerate(SCALAR_FEATURE_ORDER):
    _m = _mae_with(perm_scalar_idx=_i)
    rows.append((_name, _m[0], _m[1], _m[0] - base_mae[0]))

rows_sorted = [rows[0]] + sorted(rows[1:], key=lambda r: -r[3])
print("\n" + "=" * 78)
print("§11b  XÁO TRỘN TỪNG NGUỒN — MAE steer tăng bao nhiêu?")
print("=" * 78)
print(f"{'nguồn bị xáo':<26}{'MAE steer':>11}{'MAE long':>11}{'Δ steer':>11}   {'':<10}")
for _name, _s, _l, _d in rows_sorted:
    _bar = "#" * min(int(_d / max(base_mae[0], 1e-9) * 10), 40)
    print(f"{_name:<26}{_s:>11.5f}{_l:>11.5f}{_d:>+11.5f}   {_bar}")
print("=" * 78)

_img_gain = img_mae[0] - base_mae[0]
_scalar_gains = {r[0]: r[3] for r in rows[2:]}
_top_scalar = max(_scalar_gains, key=_scalar_gains.get) if _scalar_gains else None
IMAGE_IS_DOMINANT = bool(_top_scalar is None or _img_gain >= _scalar_gains[_top_scalar])

if _img_gain < 0.2 * base_mae[0]:
    print("KHÔNG ĐẠT: xáo toàn bộ ảnh chỉ làm MAE steer tăng "
          f"{_img_gain:+.5f} ({100*_img_gain/max(base_mae[0],1e-9):.1f}%).")
    print("  Model KHÔNG dùng ảnh segmentation để lái. Trong vòng kín nó sẽ đi thẳng cho")
    print("  tới khi ra khỏi đường, dù MAE ở §11 có đẹp đến đâu.")
elif not IMAGE_IS_DOMINANT:
    print(f"CẢNH BÁO: '{_top_scalar}' ảnh hưởng tới steer MẠNH HƠN cả ảnh "
          f"({_scalar_gains[_top_scalar]:+.5f} so với {_img_gain:+.5f}).")
    print("  Kiểm tra xem cột đó có phải hệ quả của chính hành động đang dự đoán không")
    print("  (xem LEAKY_COLS ở §2). Nếu đúng, bỏ nó rồi train lại.")
else:
    print(f"ĐẠT: ảnh là nguồn chi phối steer (Δ = {_img_gain:+.5f}, "
          f"lớn hơn mọi cột scalar).")
print("=" * 78)

# --- Nhánh phụ có thực sự học được vị trí ngang không? ------------------------------------
# Nếu aux_head không dự đoán nổi lane_offset thì đặc trưng ảnh chưa mã hoá vị trí ngang, và
# loss phụ đã không làm được việc của nó.
if AUX_LOSS_WEIGHT > 0:
    _pa, _ta = [], []
    with torch.no_grad():
        for masks, scalars, aux, _t in val_loader:
            _, pa = eval_model(masks.to(DEVICE), scalars.to(DEVICE), return_aux=True)
            _pa.append(pa.float().cpu().numpy())
            _ta.append(aux[:, AUX_TARGET_IDX].numpy())
    _pa, _ta = np.concatenate(_pa), np.concatenate(_ta)
    print("\nNhánh phụ (chỉ nhìn ẢNH) dự đoán được gì:")
    for _i, _c in enumerate(AUX_TARGET_COLS):
        _mae_aux = float(np.abs(_pa[:, _i] - _ta[:, _i]).mean())
        _base = float(np.abs(_ta[:, _i] - _ta[:, _i].mean()).mean())
        _r = float(np.corrcoef(_pa[:, _i], _ta[:, _i])[0, 1])
        print(f"  {_c:<20} MAE {_mae_aux:.4f} (hằng số {_base:.4f}, "
              f"tốt hơn {100*(1-_mae_aux/max(_base,1e-9)):+.0f}%)  corr {_r:+.3f}")
    print("  corr thấp (< 0.5) = đặc trưng ảnh vẫn chưa mã hoá vị trí ngang -> tăng")
    print("  AUX_LOSS_WEIGHT, hoặc kiểm tra lại nhãn lane_offset_m trong CSV.")


## 12. Kiểm tra sẵn sàng warm-start cho DRL


In [ ]:
# =========================================================================================
# §12. KIỂM TRA SẴN SÀNG WARM-START — chạy trước khi mang checkpoint sang DRL
# =========================================================================================
# Mọi thứ dưới đây là lỗi IM LẶNG nếu sai: shape vẫn khớp, không exception nào, chỉ có
# policy hành xử vô nghĩa sau vài nghìn step và không ai biết vì sao.
ck = torch.load(STEER_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
ready, blockers, warnings_ = True, [], []

# 0. CHỐT CHẶN RÒ RỈ — kiểm tra đầu tiên, vì nó vô hiệu hoá mọi con số còn lại.
_leaks = [c for c in (ck["continuous_cols"] + ck["raw_action_cols"]) if c in LEAKY_COLS]
if _leaks:
    ready = False
    blockers.append(
        f"Observation chứa cột RÒ RỈ: {_leaks}. Đây là hệ quả của chính hành động cần dự "
        f"đoán, đo ở cùng bước thời gian, nên model đạt MAE thấp bằng cách đọc lại đáp án. "
        f"Bỏ chúng ở §2 rồi train lại.")

# 0b. Ảnh có thực sự điều khiển steer không (kết quả của §11b).
if "IMAGE_IS_DOMINANT" in dir() and not IMAGE_IS_DOMINANT:
    ready = False
    blockers.append(
        "§11b cho thấy một cột scalar ảnh hưởng tới steer mạnh hơn cả ẢNH. Warm-start bằng "
        "checkpoint này nghĩa là đưa cho PPO một actor mù ảnh — nó sẽ phải học lái từ đầu "
        "bằng reward, tức mất hết giá trị của 34k mẫu có nhãn.")

print("HỢP ĐỒNG DRL PHẢI ĐỌC TỪ CHECKPOINT (đừng chép tay):")
print(f"  scalar_feature_order = {ck['scalar_feature_order']}")
print(f"  scalar_feature_dim   = {ck['scalar_feature_dim']}")
print(f"  traffic_light_vocab  = {ck['traffic_light_vocab']}")
print(f"  norm_stats           = "
      + ", ".join(f"{k}({v[0]:.3f},{v[1]:.3f})" for k, v in ck["norm_stats"].items()))
print(f"  control_dt           = {ck['control_dt']}  -> fixed_delta_seconds của CARLA")
print(f"  class_names          = {ck['class_names']} @ "
      f"{ck['image_height']}x{ck['image_width']} (gốc {ck['seg_native_height']}x"
      f"{ck['seg_native_width']}, thin_cover={ck['thin_cover_thresh']})")

# 1. Ngưỡng quyết định — CHỌN baseline theo hợp đồng observation, không cố định.
_has_prev = len(ck["raw_action_cols"]) > 0
if _has_prev:
    # Model được nhìn previous_steer -> baseline "chép" là đối thủ công bằng.
    if not (ck["val_steer_mae"] < ck["copycat_steer_mae"]):
        ready = False
        blockers.append(
            f"steer MAE {ck['val_steer_mae']:.5f} KHÔNG thấp hơn baseline chép "
            f"{ck['copycat_steer_mae']:.5f}. Model đã học ánh xạ 'output ~ previous_steer', "
            f"một điểm hút mà PPO/SAC phải phá bỏ trước khi học được gì.")
else:
    # v5: model KHÔNG có previous_steer trong observation. So nó với baseline "chép" là so
    # một thí sinh bị bịt mắt với một thí sinh được xem đáp án — thua là chuyện đương nhiên
    # và không nói lên điều gì về năng lực lái. Ngưỡng công bằng là baseline HẰNG SỐ.
    if not (ck["val_steer_mae"] < ck["const_steer_mae"]):
        ready = False
        blockers.append(
            f"steer MAE {ck['val_steer_mae']:.5f} KHÔNG thấp hơn baseline hằng số "
            f"{ck['const_steer_mae']:.5f} (luôn dự đoán 0). Model chưa học được gì từ ảnh.")
    print(f"\n[i] Baseline 'chép prev' ({ck['copycat_steer_mae']:.5f}) chỉ để tham khảo: v5 đã")
    print("    bỏ previous_steer khỏi observation nên model không thể — và không cần — vượt")
    print("    nó. Ngưỡng thật sự là §11b: ẢNH phải là nguồn chi phối steer.")

# 2. Ô one-hot chết: chiều nào của scalar_mlp gần như không được huấn luyện?
_w = ck["model_state_dict"]["scalar_mlp.0.weight"].abs().mean(dim=0)
for _i, _name in enumerate(ck["scalar_feature_order"]):
    if _w[_i] < 0.02 * _w.mean():
        warnings_.append(f"'{_name}' có trọng số vào scalar_mlp gần như bằng 0 — nhiều khả "
                         f"năng đặc trưng này không bao giờ bật trong dữ liệu train.")

# 3. Vùng lệch làn — nơi vòng kín thật sự sống hay chết.
if not np.isnan(RECOVERY_MAE) and RECOVERY_MAE > 3 * CENTER_MAE:
    warnings_.append(
        f"MAE lệch làn ({RECOVERY_MAE:.4f}) gấp {RECOVERY_MAE/CENTER_MAE:.1f} lần MAE giữa "
        f"làn ({CENTER_MAE:.4f}). Policy sẽ ổn khi đi đúng và lạc lối ngay khi trôi -> sai "
        f"số tự khuếch đại. Ưu tiên SAMPLER_MODE='offset' hoặc thu thêm pha recovery.")

# 4. Lệch phân phối seg giữa lúc train IL và lúc chạy DRL.
# Dự án hiện KHÔNG dùng model segmentation trong vòng suy luận: `drl_training` và bridge
# server đều đọc thẳng camera `sensor.camera.semantic_segmentation` của CARLA rồi remap qua
# cùng một LUT 4 lớp. Nên khi IL cũng học trên ground-truth thì hai bên KHỚP nhau.
if ck["trained_on_predicted_seg"]:
    warnings_.append("IL học trên mask DỰ ĐOÁN. Đảm bảo DRL/bridge cũng chạy trên mask dự "
                     "đoán chứ không phải camera segmentation ground-truth của CARLA, nếu "
                     "không thì đây mới chính là chỗ lệch phân phối.")
else:
    print("\n[i] IL học trên mask GROUND-TRUTH, và đường chạy thật (drl_training + bridge)")
    print("    cũng đọc camera segmentation ground-truth của CARLA -> KHÔNG có lệch phân")
    print("    phối. Lệch chỉ xuất hiện nếu sau này thay bằng đầu ra của model segmentation.")

# 4b. Xe có tự cất bánh được không (kết quả của §11c).
if "STARTS_FROM_STOP" in globals() and not STARTS_FROM_STOP:
    ready = False
    blockers.append(
        "§11c: model gần như không bao giờ đạp ga khi xe đứng yên. Mọi episode đều bắt đầu "
        "ở tốc độ 0, nên xe sẽ không rời vạch xuất phát — đúng cách checkpoint v8 đã hỏng. "
        "Bật DROP_STATIONARY_RUNS ở §2 (hoặc giảm STATIONARY_KEEP) rồi train lại.")

# 5. Lỗi đổi dấu.
if brake_as_throttle > 0.005 * len(all_preds):
    warnings_.append(f"{brake_as_throttle} frame 'GT phanh -> model đạp ga' "
                     f"({100*brake_as_throttle/len(all_preds):.2f}%). MAE không thấy nhưng "
                     f"trong vòng kín đây là va chạm.")

print("\n" + "=" * 74)
for b in blockers:
    print("CHẶN   : " + b.replace("\n", "\n         "))
for w in warnings_:
    print("LƯU Ý  : " + w.replace("\n", "\n         "))
if ready:
    print("\nSẴN SÀNG warm-start. Bốn thứ phải làm đúng bên DRL (đã đặt sẵn trong")
    print("drl_training/ppo_config.json — liệt kê ở đây để đối chiếu):")
    print("  1. Nhịp ra quyết định = {:.2f}s. KHÔNG hạ fixed_delta_seconds lên {:.2f}s —"
          .format(ck["control_dt"], ck["control_dt"]))
    print("     CARLA khuyến cáo <= 0.05s, trên mức đó vật lý bắt đầu sai. Dùng")
    print("     fps = 20 (delta 0.05s) + action_repeat = {:d}."
          .format(int(round(ck["control_dt"] / 0.05))))
    print("  2. lane_offset_m / heading_error_rad chỉ dùng cho REWARD (và cho LOSS PHỤ ở §8),")
    print("     tuyệt đối không đưa vào observation.")
    print("  3. Critic khởi tạo ngẫu nhiên sẽ sinh advantage nhiễu và XOÁ trọng số IL trong")
    print("     vài trăm update đầu. Bắt buộc: critic_warmup_updates > 0 (đóng băng actor),")
    print("     actor_lr 1e-5..3e-5, critic_lr 3e-4.")
    print("  4. log_std khởi tạo THEO TỪNG CHIỀU: [-3.0, -1.5] -> std 0.050 / 0.223. Dùng")
    print("     chung một giá trị cho cả hai chiều (vd -1.2 -> std 0.30) là nhiễu gấp 10-60")
    print("     lần biên độ lái thật, đủ để lăng xe ra khỏi làn ngay rollout đầu tiên.")
else:
    print("\nCHƯA sẵn sàng. Thử theo thứ tự, mỗi lần ĐỔI MỘT thứ và đổi RUN_NAME:")
    print("  1. AUX_LOSS_WEIGHT 0.5 -> 1.0 rồi 2.0 — ép nhánh CNN mã hoá vị trí ngang mạnh")
    print("     hơn. Đây là knob trực tiếp nhất, thử trước tiên.")
    print("  2. SAMPLER_MODE = 'offset'      — tăng tần suất frame đã lệch làn")
    print("  3. POOL_GRID (4,6) -> (6,8)     — giữ bố cục không gian chi tiết hơn.")
    print("     NHỚ sửa đồng bộ drl_training/policy/backbone.py, nếu không warm-start báo lỗi.")
    print("  4. Thu thêm dữ liệu hồi phục (DAgger): đặt xe lệch 0.3-0.8m rồi ghi lại lúc")
    print("     autopilot lái về. Tốn nhất nhưng chắc chắn nhất.")
print("=" * 74)
del ck
